In [ ]:

import sys
from pathlib import Path
import torch

print("PYTHON:", sys.executable)
print("PLATFORM:", sys.platform)
print("KAGGLE_WORKING:", Path("/kaggle/working").is_dir())
print("CUDA_AVAILABLE:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import sys
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

print("python =", sys.executable)
print("input exists =", INPUT.is_dir())
print("working exists =", WORK.is_dir())

zips = list(INPUT.rglob("kaggle_source_warm2_v1.zip"))
print("zip files:", zips)

for p in INPUT.iterdir():
    print("TOP:", p)

In [ ]:

from pathlib import Path

INPUT = Path("/kaggle/input")

print("========== KAGGLE INPUT ==========")

for root in sorted(INPUT.iterdir()):
    print(f"\nDATASET: {root}")

    paths = sorted(root.rglob("*"))
    print("TOTAL_ENTRIES:", len(paths))

    for path in paths[:60]:
        relative = path.relative_to(root)

        if path.is_file():
            print("FILE:", relative, "bytes=", path.stat().st_size)
        elif path.is_dir():
            print("DIR: ", relative)

print("\n========== SOURCE DISCOVERY ==========")

names = [
    "kaggle_source_warm2_v1.zip",
    "bundle_manifest.json",
    "edge_predictor_best.pth",
    "frozen_v9.py",
]

for name in names:
    matches = sorted(INPUT.rglob(name))
    print(f"\n{name}: {len(matches)} match(es)")

    for path in matches:
        print(" ", path)

print("\n14.15A3_INPUT_DISCOVERY_COMPLETE")

In [ ]:

# Notebook 14 / 14.15A-3
# Verify the mounted source bundle and inspect installed dependencies.
# Read-only; no inference, GT, evaluator, or installation.

from pathlib import Path, PurePosixPath
from importlib import metadata
import hashlib
import json
import sys

ROOT = Path(
    "/kaggle/input/datasets/markdd520/"
    "biohub-frozen-v9-warm2-bundle/bundle"
)

EXPECTED_MANIFEST_SHA = (
    "615c3550cba46a5032344549d2769201af22c0425beb9349726874669ae9c865"
)

def require(ok, message):
    if not ok:
        raise RuntimeError(message)

def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("========== 14.15A-3 / ENVIRONMENT ==========")

require(sys.platform == "linux", "当前不是 Linux")
require(
    Path("/kaggle/working").is_dir(),
    "当前不是 Kaggle 运行环境",
)

print("PYTHON =", sys.executable)
print("ROOT =", ROOT)

require(
    ROOT.is_dir() and not ROOT.is_symlink(),
    "源码包根目录不存在或为符号链接",
)

manifest_path = ROOT / "bundle_manifest.json"

require(
    manifest_path.is_file() and not manifest_path.is_symlink(),
    "bundle_manifest.json 不存在",
)

actual_manifest_sha = sha256_file(manifest_path)

require(
    actual_manifest_sha == EXPECTED_MANIFEST_SHA,
    f"Manifest SHA 不匹配：{actual_manifest_sha}",
)

with manifest_path.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

require(
    manifest.get("status") == "SOURCE_AND_WEIGHTS_BUNDLE_ONLY",
    "Manifest 状态不符合预期",
)

files = manifest.get("files")

require(
    isinstance(files, dict) and len(files) == 29,
    "Manifest 文件清单不是预期的 29 个文件",
)

print("MANIFEST_SHA256 =", actual_manifest_sha)
print("MANIFEST_FILE_COUNT =", len(files))

print("\n========== FILE INTEGRITY ==========")

for relative, info in sorted(files.items()):
    rel = PurePosixPath(relative)

    require(
        not rel.is_absolute()
        and ".." not in rel.parts
        and relative == rel.as_posix(),
        f"非法相对路径：{relative}",
    )

    path = ROOT.joinpath(*rel.parts)

    require(
        path.is_file() and not path.is_symlink(),
        f"文件缺失或为符号链接：{relative}",
    )

    require(
        path.stat().st_size == info["bytes"],
        f"文件大小不符：{relative}",
    )

    actual = sha256_file(path)

    require(
        actual == info["sha256"],
        f"文件 SHA 不匹配：{relative}",
    )

    print("PASS", relative)

# Ensure no unexpected files have been inserted into the mounted bundle.
actual_files = {
    p.relative_to(ROOT).as_posix()
    for p in ROOT.rglob("*")
    if p.is_file()
}

expected_files = set(files) | {"bundle_manifest.json"}

require(
    actual_files == expected_files,
    "挂载后的文件集合与 manifest 不一致："
    f"missing={sorted(expected_files - actual_files)}, "
    f"extra={sorted(actual_files - expected_files)}",
)

print("\nBUNDLE_SHA_AND_FILE_SET=PASS")

print("\n========== KAGGLE DEPENDENCIES ==========")

packages = [
    "torch",
    "numpy",
    "pandas",
    "scipy",
    "polars",
    "zarr",
    "numcodecs",
    "geff",
    "tracksdata",
    "dask",
    "blosc2",
    "scikit-image",
    "rustworkx",
    "networkx",
    "numba",
    "tqdm",
    "pyarrow",
    "imagecodecs",
    "ilpy",
]

missing_packages = []

for name in packages:
    try:
        version = metadata.version(name)
    except metadata.PackageNotFoundError:
        version = "NOT_INSTALLED"
        missing_packages.append(name)

    print(f"{name:16s} {version}")

print("\nMISSING_PACKAGES =", missing_packages)

print("\n========== DATA MOUNT ==========")

INPUT = Path("/kaggle/input")

for entry in sorted(INPUT.iterdir()):
    print("INPUT:", entry)

competition_candidates = [
    INPUT / "competitions" / "biohub-cell-tracking-during-development",
    INPUT / "biohub-cell-tracking-during-development",
]

competition_mounts = [
    p for p in competition_candidates if (p / "test").is_dir()
]

print("COMPETITION_MOUNTS =", competition_mounts)

print("\n14.15A3_BUNDLE_INTEGRITY_PASS")
print("RUNTIME_IMPORT_VERIFIED=NO")
print("INFERENCE_RUN=NO")
print("FILES_WRITTEN=0")
print("GT_READ=NO")
print("KAGGLE_SUBMISSION=NO")

In [ ]:

# Notebook 14 / 14.15A-4
# Competition input discovery — read-only.

from pathlib import Path
import sys

assert sys.platform == "linux"
assert Path("/kaggle/working").is_dir()

INPUT = Path("/kaggle/input")

print("========== INPUT ROOTS ==========")

for p in sorted(INPUT.iterdir()):
    print("ROOT:", p)

print("\n========== COMPETITION DISCOVERY ==========")

samples = sorted(INPUT.rglob("sample_submission.csv"))

for sample in samples:
    candidate = sample.parent

    # Ignore the sample file inside our source bundle.
    if candidate.name != "biohub-cell-tracking-during-development":
        continue

    test_dir = candidate / "test"
    train_dir = candidate / "train"

    print("CANDIDATE:", candidate)
    print("TEST_EXISTS:", test_dir.is_dir())
    print("TRAIN_EXISTS:", train_dir.is_dir())

    if test_dir.is_dir():
        test_ids = sorted(
            p.stem for p in test_dir.glob("*.zarr")
        )

        print("TEST_COUNT:", len(test_ids))
        print("TEST_IDS:", test_ids)

        print(
            "TEST_ZARR_PRESENT:",
            all(
                (test_dir / f"{name}.zarr").is_dir()
                for name in test_ids
            )
        )

print("\n14.15A4_INPUT_DISCOVERY_COMPLETE")
print("GT_READ=NO")
print("FILES_WRITTEN=0")

In [ ]:

# Notebook 14 / 14.15A-4-R1
# Read-only competition mount discovery.

from pathlib import Path
import sys

assert sys.platform == "linux"
assert Path("/kaggle/working").is_dir()

BASE = Path("/kaggle/input/competitions")
EXPECTED_NAME = "biohub-cell-tracking-during-development"

print("========== COMPETITION MOUNT ==========")
print("BASE_EXISTS =", BASE.is_dir())

if BASE.is_dir():
    for entry in sorted(BASE.iterdir()):
        print("ENTRY =", entry)
        print("  IS_DIR =", entry.is_dir())
        print("  IS_FILE =", entry.is_file())

print("\n========== KNOWN PATH CHECK ==========")

candidates = [
    BASE / EXPECTED_NAME,
    Path("/kaggle/input") / EXPECTED_NAME,
]

for candidate in candidates:
    print("\nCANDIDATE =", candidate)
    print("EXISTS =", candidate.exists())
    print("IS_DIR =", candidate.is_dir())

    if not candidate.is_dir():
        continue

    # Inspect immediate children only: do not recursively traverse train/zarr.
    for child in sorted(candidate.iterdir()):
        print(
            "  CHILD =", child.name,
            "DIR =", child.is_dir(),
            "FILE =", child.is_file(),
        )

    test_dir = candidate / "test"

    if test_dir.is_dir():
        test_zarrs = sorted(test_dir.glob("*.zarr"))
        test_ids = [p.stem for p in test_zarrs]

        print("  TEST_DIR =", test_dir)
        print("  TEST_COUNT =", len(test_ids))
        print("  TEST_IDS =", test_ids)
        print(
            "  SAMPLE_EXISTS =",
            (candidate / "sample_submission.csv").is_file(),
        )

print("\n14.15A4_R1_MOUNT_DISCOVERY_COMPLETE")
print("FILES_WRITTEN=0")
print("GT_READ=NO")

In [ ]:

# Notebook 14 / 14.15A-5
# Network capability probe — read-only.

import sys
import urllib.request
import urllib.error
from pathlib import Path

assert sys.platform == "linux"
assert Path("/kaggle/working").is_dir()

targets = {
    "PYPI": "https://pypi.org/simple/geff/",
    "GITHUB": "https://github.com/royerlab/tracksdata",
}

print("========== 14.15A-5 / NETWORK ==========")

for name, url in targets.items():
    try:
        request = urllib.request.Request(
            url,
            method="HEAD",
            headers={"User-Agent": "BioHub-Kaggle-Dependency-Probe"},
        )

        with urllib.request.urlopen(request, timeout=8) as response:
            print(
                f"{name}: REACHABLE "
                f"HTTP={response.status}"
            )

    except Exception as exc:
        print(
            f"{name}: NOT_CONFIRMED "
            f"{type(exc).__name__}: {exc}"
        )

print("\n14.15A5_NETWORK_PROBE_COMPLETE")
print("PACKAGES_INSTALLED=0")
print("FILES_WRITTEN=0")
print("GT_READ=NO")
print("INFERENCE_RUN=NO")

In [ ]:

# Notebook 14 / 14.15A-6
# Install missing packages in the interactive Kaggle session.
# Do not upgrade the preinstalled CUDA/PyTorch stack.

import sys
import subprocess
from pathlib import Path
from importlib import metadata

assert sys.platform == "linux"
assert Path("/kaggle/working").is_dir()

import torch

assert torch.cuda.is_available(), "Kaggle GPU 不可用"

torch_before = torch.__version__
numpy_before = metadata.version("numpy")
pandas_before = metadata.version("pandas")

print("========== FROZEN ENVIRONMENT ==========")
print("PYTHON =", sys.executable)
print("TORCH =", torch_before)
print("NUMPY =", numpy_before)
print("PANDAS =", pandas_before)

# Match the previously recorded local versions where possible.
# --no-deps prevents pip from silently replacing CUDA/PyTorch,
# NumPy or other preinstalled packages.
packages = [
    "zarr==3.1.6",
    "numcodecs==0.15.1",
    "geff==1.3.0.1.2",
    "rustworkx==0.18.1",
    "imagecodecs==2026.3.6",
    "ilpy==0.6.0",
]

def run_install(label, command):
    print(f"\n========== {label} ==========", flush=True)
    print("COMMAND =", command, flush=True)

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        errors="replace",
    )

    print(result.stdout[-16000:], flush=True)
    print("EXIT_CODE =", result.returncode, flush=True)

    if result.returncode != 0:
        raise RuntimeError(
            f"{label} FAILED；停止，不自动修改版本或重试"
        )

run_install(
    "INSTALL BINARY DEPENDENCIES",
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps",
        "--only-binary=:all:",
        *packages,
    ],
)

# Exact Git commit from the frozen local tracksdata dependency.
tracksdata_rev = (
    "7bfeaf845ceb951226f19b72fe5b80e01601018a"
)

run_install(
    "INSTALL PINNED TRACKSDATA",
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps",
        "git+https://github.com/royerlab/tracksdata.git@"
        + tracksdata_rev,
    ],
)

print("\n========== POST-INSTALL VERSIONS ==========")

for name in [
    "torch", "numpy", "pandas",
    "zarr", "numcodecs", "geff",
    "rustworkx", "imagecodecs", "ilpy",
    "tracksdata",
]:
    print(name, "=", metadata.version(name))

assert metadata.version("torch") == torch_before
assert metadata.version("numpy") == numpy_before
assert metadata.version("pandas") == pandas_before

tracksdata_version = metadata.version("tracksdata")

assert "g7bfeaf845" in tracksdata_version, (
    "tracksdata 的版本标识与冻结 Git commit 不一致："
    + tracksdata_version
)

print("\n14.15A6_DEPENDENCY_INSTALL_COMPLETED")
print("CUDA_TORCH_UNCHANGED=YES")
print("RUNTIME_IMPORT_VERIFIED=NO")
print("OFFLINE_DEPENDENCIES_READY=NO")
print("GT_READ=NO")
print("INFERENCE_RUN=NO")

In [ ]:

# Notebook 14 / 14.15A-7-R1
# Rediscover the frozen source bundle after a Kaggle mount change.

from pathlib import Path
import hashlib
import json
import sys

EXPECTED_MANIFEST_SHA = (
    "615c3550cba46a5032344549d2769201af22c0425beb9349726874669ae9c865"
)

def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

print("========== RUNTIME ==========")
print("PYTHON =", sys.executable)
print("PLATFORM =", sys.platform)

assert sys.platform == "linux"
assert Path("/kaggle/working").is_dir()

INPUT = Path("/kaggle/input")

print("\n========== CURRENT MOUNTS ==========")

for path in sorted(INPUT.iterdir()):
    print("INPUT =", path)

    if path.is_dir():
        for child in sorted(path.iterdir()):
            print("  CHILD =", child)

print("\n========== SOURCE DISCOVERY ==========")

candidates = sorted(INPUT.rglob("bundle_manifest.json"))
matches = []

for manifest_path in candidates:
    actual_sha = sha256_file(manifest_path)

    print("CANDIDATE =", manifest_path)
    print("SHA256 =", actual_sha)

    if actual_sha == EXPECTED_MANIFEST_SHA:
        matches.append(manifest_path)

print("MATCH_COUNT =", len(matches))

if len(matches) != 1:
    print("14.15A7_SOURCE_MOUNT_BLOCKED")
    print("需要在当前 Notebook 中重新附加正确的源码 Dataset。")
else:
    ROOT = matches[0].parent
    manifest = json.loads(matches[0].read_text())

    assert len(manifest["files"]) == 29

    # Verify every packaged file, not just the manifest.
    for relative, info in manifest["files"].items():
        path = ROOT / relative

        assert path.is_file() and not path.is_symlink(), relative
        assert path.stat().st_size == info["bytes"], relative
        assert sha256_file(path) == info["sha256"], relative

    print("ROOT =", ROOT)
    print("FILES_VERIFIED =", len(manifest["files"]))
    print("14.15A7_SOURCE_MOUNT_RECOVERED")

print("FILES_WRITTEN=0")
print("GT_READ=NO")
print("INFERENCE_RUN=NO")

In [ ]:

from pathlib import Path
import hashlib

EXPECTED = "615c3550cba46a5032344549d2769201af22c0425beb9349726874669ae9c865"

matches = []

for p in Path("/kaggle/input/datasets").rglob("bundle_manifest.json"):
    digest = hashlib.sha256(p.read_bytes()).hexdigest()
    print("CANDIDATE =", p)
    print("SHA256 =", digest)
    if digest == EXPECTED:
        matches.append(p)

print("MATCH_COUNT =", len(matches))

assert len(matches) == 1, "源码 Dataset 尚未正确挂载"

ROOT = matches[0].parent

assert (ROOT / "scripts/predict_unet_transformer_mps.py").is_file()
assert (ROOT / "src/biohub_cell_tracking/frozen_v9.py").is_file()

print("ROOT =", ROOT)
print("14.15A7_SOURCE_MOUNT_RECOVERED")

In [ ]:

from pathlib import Path

BASE = Path("/kaggle/input/datasets/markdd520")

print("BASE_EXISTS =", BASE.exists())

if BASE.is_dir():
    for dataset in sorted(BASE.iterdir()):
        print("\nDATASET =", dataset)
        print("IS_DIR =", dataset.is_dir())

        if dataset.is_dir():
            for item in sorted(dataset.iterdir()):
                print("  ENTRY =", item.name)

                if item.is_dir():
                    for child in sorted(item.iterdir()):
                        print("    CHILD =", child.name)

print("\n14.15A7_MOUNT_STRUCTURE_INSPECTED")

In [ ]:

from pathlib import Path
import hashlib

base = Path("/kaggle/input/datasets/markdd520")

print("DATASETS:", [p.name for p in base.iterdir()])

manifests = list(
    Path("/kaggle/input").rglob("bundle_manifest.json")
)

print("MANIFESTS:", manifests)

expected = "615c3550cba46a5032344549d2769201af22c0425beb9349726874669ae9c865"

for p in manifests:
    actual = hashlib.sha256(p.read_bytes()).hexdigest()
    print("SHA_MATCH:", actual == expected)
    if actual == expected:
        ROOT = p.parent
        print("ROOT:", ROOT)
        print("14.15A7_SOURCE_MOUNT_RECOVERED")

In [ ]:

# Notebook 14 / 14.15A-8-R1
# Install only the missing zarr dependency.

import sys
import subprocess
from importlib import metadata
from pathlib import Path

assert sys.platform == "linux"
assert Path("/kaggle/working").is_dir()

torch_before = metadata.version("torch")
numpy_before = metadata.version("numpy")
pandas_before = metadata.version("pandas")

print("========== INSTALL DONFIG ==========", flush=True)

cmd = [
    sys.executable, "-m", "pip", "install",
    "--no-deps",
    "--only-binary=:all:",
    "donfig==0.8.1.post1",
]

result = subprocess.run(
    cmd,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(result.stdout[-12000:], flush=True)
print("EXIT_CODE =", result.returncode)

assert result.returncode == 0, "donfig 安装失败，停止"

assert metadata.version("torch") == torch_before
assert metadata.version("numpy") == numpy_before
assert metadata.version("pandas") == pandas_before

import donfig
import zarr

print("\n========== IMPORT CHECK ==========")
print("DONFIG =", metadata.version("donfig"))
print("ZARR =", metadata.version("zarr"))
print("TORCH_UNCHANGED =", metadata.version("torch") == torch_before)

print("\n14.15A8_R1_DONFIG_IMPORT_PASS")
print("MODEL_INFERENCE_RUN=NO")
print("GT_READ=NO")

In [ ]:

# Notebook 14 / 14.15A-8-R2
# Resolve the missing geff_spec dependency from installed metadata.

import sys
import subprocess
import importlib
from pathlib import Path
from importlib import metadata
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name

assert sys.platform == "linux"
assert Path("/kaggle/working").is_dir()

def version(name):
    return metadata.version(name)

before = {
    name: version(name)
    for name in ("torch", "numpy", "pandas", "geff", "zarr")
}

print("========== GEFF DEPENDENCY METADATA ==========")

requirements = metadata.requires("geff") or []
target = []

for raw in requirements:
    req = Requirement(raw)
    print("REQUIRES:", raw)

    if canonicalize_name(req.name) == "geff-spec":
        if req.marker is None or req.marker.evaluate({"extra": ""}):
            target.append(str(req))

assert len(target) == 1, (
    "无法从 geff 元数据确认唯一的 geff-spec 依赖；停止"
)

dependency = target[0]
print("SELECTED_REQUIREMENT =", dependency)

print("\n========== INSTALL MISSING DEPENDENCY ==========")

cmd = [
    sys.executable, "-m", "pip", "install",
    "--no-deps",
    "--only-binary=:all:",
    dependency,
]

result = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    errors="replace",
)

print(result.stdout[-12000:])
print("EXIT_CODE =", result.returncode)

assert result.returncode == 0, "geff-spec 安装失败，停止"

for name, expected in before.items():
    assert version(name) == expected, f"{name} 版本发生变化"

importlib.invalidate_caches()

print("\n========== IMPORT CHECK ==========")

geff_spec = importlib.import_module("geff_spec")
print("GEFF_SPEC_VERSION =", version("geff-spec"))
print("GEFF_SPEC_PATH =", geff_spec.__file__)

geff = importlib.import_module("geff")
print("GEFF_VERSION =", version("geff"))
print("GEFF_PATH =", geff.__file__)

print("\n14.15A8_R2_GEFF_IMPORT_PASS")
print("TORCH_UNCHANGED=YES")
print("GT_READ=NO")
print("INFERENCE_RUN=NO")

In [ ]:

# Notebook 14 / 14.15A-8-R3
# Align Polars and its binary runtime with the successful local environment.

import sys
import subprocess
from pathlib import Path
from importlib import metadata

assert sys.platform == "linux"
assert Path("/kaggle/working").is_dir()

TARGET = "1.43.2"

protected = {
    name: metadata.version(name)
    for name in ("torch", "numpy", "pandas", "tracksdata", "geff")
}

print("========== PRECHECK ==========")

print("POLARS_BEFORE =", metadata.version("polars"))

runtime_names = [
    "polars-runtime-32",
    "polars-runtime-64",
    "polars-runtime-compat",
]

installed_runtimes = {}

for name in runtime_names:
    try:
        installed_runtimes[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        pass

print("INSTALLED_RUNTIMES =", installed_runtimes)

assert metadata.version("polars") == "1.35.2", (
    "Polars 已发生变化，不重复执行安装"
)

assert len(installed_runtimes) == 1, (
    "无法确定唯一的 Polars binary runtime，停止"
)

runtime_name = next(iter(installed_runtimes))

print("SELECTED_RUNTIME =", runtime_name)

print("\n========== INSTALL ==========", flush=True)

cmd = [
    sys.executable, "-m", "pip", "install",
    "--no-deps",
    "--only-binary=:all:",
    f"polars=={TARGET}",
    f"{runtime_name}=={TARGET}",
]

print("COMMAND =", cmd, flush=True)

result = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    errors="replace",
)

print(result.stdout[-16000:], flush=True)
print("EXIT_CODE =", result.returncode)

assert result.returncode == 0, (
    "Polars 安装失败；停止，不自动更换其他版本"
)

print("\n========== POST-INSTALL METADATA ==========")

print("POLARS =", metadata.version("polars"))
print("RUNTIME =", metadata.version(runtime_name))

assert metadata.version("polars") == TARGET
assert metadata.version(runtime_name) == TARGET

for name, previous_version in protected.items():
    current_version = metadata.version(name)

    assert current_version == previous_version, (
        f"受保护依赖发生变化：{name}, "
        f"{previous_version} -> {current_version}"
    )

print("\n14.15A8_R3_POLARS_INSTALL_PASS")
print("PROTECTED_PACKAGE_VERSIONS=UNCHANGED")
print("KERNEL_RESTART_REQUIRED=YES")
print("GT_READ=NO")
print("INFERENCE_RUN=NO")

In [ ]:

# Notebook 14 / 14.15A-8-R4
# Verify the Polars fix after kernel restart.

import sys
from pathlib import Path
from importlib import metadata

assert sys.platform == "linux"
assert Path("/kaggle/working").is_dir()

print("========== PACKAGE VERSIONS ==========")

for name in ("torch", "numpy", "polars", "polars-runtime-32",
             "tracksdata", "geff", "zarr"):
    print(name, "=", metadata.version(name))

import polars as pl

print("POLARS_FLOAT16_AVAILABLE =", hasattr(pl, "Float16"))
assert metadata.version("polars") == "1.43.2"
assert hasattr(pl, "Float16"), "Polars Float16 仍不可用"

import tracksdata

print("TRACKSDATA_PATH =", tracksdata.__file__)
print("TRACKSDATA_VERSION =", metadata.version("tracksdata"))

assert metadata.version("tracksdata") == "0.1.0rc9.dev4+g7bfeaf845"

print("\n14.15A8_R4_TRACKSDATA_IMPORT_PASS")
print("GT_READ=NO")
print("INFERENCE_RUN=NO")

In [ ]:

# Notebook 14 / 14.15A-8-R5
# Install the missing bidict dependency and inspect tracksdata requirements.

import sys
import subprocess
import importlib
from pathlib import Path
from importlib import metadata

assert sys.platform == "linux"
assert Path("/kaggle/working").is_dir()

protected = {
    name: metadata.version(name)
    for name in (
        "torch", "numpy", "pandas",
        "polars", "tracksdata", "geff", "zarr",
    )
}

print("========== TRACKSDATA DEPENDENCY DECLARATIONS ==========")

for requirement in metadata.requires("tracksdata") or []:
    print("REQUIRES:", requirement)

print("\n========== INSTALL BIDICT ==========", flush=True)

command = [
    sys.executable, "-m", "pip", "install",
    "--no-deps",
    "--only-binary=:all:",
    "bidict==0.23.1",
]

result = subprocess.run(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    errors="replace",
)

print(result.stdout[-12000:], flush=True)
print("EXIT_CODE =", result.returncode)

assert result.returncode == 0, "bidict 安装失败，停止"

for name, expected in protected.items():
    actual = metadata.version(name)
    assert actual == expected, (
        f"受保护依赖发生变化：{name}: {expected} -> {actual}"
    )

importlib.invalidate_caches()

import bidict

print("\nBIDICT_VERSION =", metadata.version("bidict"))
print("BIDICT_PATH =", bidict.__file__)

print("\n========== PIP DEPENDENCY CHECK ==========", flush=True)

check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    errors="replace",
)

print(check.stdout[-16000:])
print("PIP_CHECK_EXIT_CODE =", check.returncode)

print("\n14.15A8_R5_BIDICT_INSTALL_PASS")
print("PROTECTED_PACKAGES_UNCHANGED=YES")
print("TRACKSDATA_IMPORT_VERIFIED=NO")
print("GT_READ=NO")
print("INFERENCE_RUN=NO")

In [ ]:

# Notebook 14 / 14.15A-8-R6
# Read-only dependency and import diagnostics.

import sys
import importlib
import importlib.util
import traceback
from pathlib import Path
from importlib import metadata
from packaging.requirements import Requirement

assert sys.platform == "linux"
assert Path("/kaggle/working").is_dir()

ROOT = Path(
    "/kaggle/input/datasets/markdd520/"
    "biohub-frozen-v9-warm2-bundle/bundle"
)

assert ROOT.is_dir(), "源码 Dataset 未挂载"

sys.dont_write_bytecode = True

# Keep the verified source bundle first in the import search path.
project_paths = [
    ROOT / "external/official/scripts",
    ROOT / "external/official/src",
    ROOT / "src",
    ROOT / "scripts",
]

for path in reversed(project_paths):
    assert path.is_dir(), f"缺少目录：{path}"
    value = str(path)
    if value in sys.path:
        sys.path.remove(value)
    sys.path.insert(0, value)

print("========== CORE DEPENDENCY AUDIT ==========")

missing = []
incompatible = []

for owner in ("tracksdata", "geff", "ilpy"):
    print(f"\nPACKAGE={owner}")

    for raw in metadata.requires(owner) or []:
        req = Requirement(raw)

        # Ignore optional extras not activated by this installation.
        if req.marker is not None:
            if not req.marker.evaluate({"extra": ""}):
                continue

        try:
            installed = metadata.version(req.name)
        except metadata.PackageNotFoundError:
            installed = "NOT_INSTALLED"
            missing.append((owner, str(req)))
        else:
            if req.specifier and installed not in req.specifier:
                incompatible.append((owner, str(req), installed))

        print(f"  {req} -> {installed}")

print("\nMISSING_CORE_DEPENDENCIES =", missing)
print("INCOMPATIBLE_CORE_DEPENDENCIES =", incompatible)

print("\n========== TARGETED IMPORT TESTS ==========")

modules = [
    "bidict",
    "psygnal",
    "rich",
    "sqlalchemy",
    "pyscipopt",
    "ilpy",
    "geff",
    "tracksdata",
    "tracking_cellmot.io",
    "tracking_cellmot.metrics",
    "train_unet_transformer_mps",
    "predict_unet_transformer_mps",
    "biohub_cell_tracking.frozen_v9",
    "geffs_to_csv",
    "csv_to_geffs",
]

import_results = {}

for name in modules:
    try:
        module = importlib.import_module(name)

        origin = getattr(module, "__file__", None)

        # A module with this name must not silently resolve
        # to some unrelated external copy.
        if name in (
            "tracking_cellmot.io",
            "tracking_cellmot.metrics",
            "train_unet_transformer_mps",
            "predict_unet_transformer_mps",
            "biohub_cell_tracking.frozen_v9",
            "geffs_to_csv",
            "csv_to_geffs",
        ):
            if origin is None:
                raise RuntimeError("项目模块缺少 __file__")

            if not Path(origin).resolve().is_relative_to(ROOT.resolve()):
                raise RuntimeError(
                    f"项目模块来源错误：{origin}"
                )

        import_results[name] = "PASS"
        print(f"PASS {name}: {origin}", flush=True)

    except Exception as exc:
        import_results[name] = (
            f"{type(exc).__name__}: {exc}"
        )

        print(f"FAIL {name}: {import_results[name]}", flush=True)

print("\n========== SUMMARY ==========")

failed = {
    name: result
    for name, result in import_results.items()
    if result != "PASS"
}

print("FAILED_IMPORTS =", failed)
print("MISSING_CORE_DEPENDENCIES =", missing)
print("INCOMPATIBLE_CORE_DEPENDENCIES =", incompatible)

if not failed and not missing and not incompatible:
    print("14.15A8_R6_CORE_IMPORTS_PASS")
else:
    print("14.15A8_R6_DEPENDENCIES_INCOMPLETE")

print("INFERENCE_RUN=NO")
print("GT_READ=NO")
print("EVALUATOR_RUN=NO")
print("PACKAGES_INSTALLED=0")

In [ ]:

# Notebook 14 / 14.15A-8-R7
# Complete the remaining declared core dependency.

import sys
import subprocess
import importlib
from pathlib import Path
from importlib import metadata

assert sys.platform == "linux"
assert Path("/kaggle/working").is_dir()

PROTECTED = (
    "torch",
    "numpy",
    "pandas",
    "polars",
    "polars-runtime-32",
    "tracksdata",
    "geff",
    "zarr",
    "ilpy",
)

before = {name: metadata.version(name) for name in PROTECTED}

assert before["torch"] == "2.10.0+cu128"
assert before["tracksdata"] == "0.1.0rc9.dev4+g7bfeaf845"
assert before["polars"] == "1.43.2"

print("========== INSTALL PYSCIPOPT ==========", flush=True)

command = [
    sys.executable,
    "-m", "pip", "install",
    "--no-deps",
    "--only-binary=:all:",
    "pyscipopt==6.2.1",
]

result = subprocess.run(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    errors="replace",
)

print(result.stdout[-16000:], flush=True)
print("EXIT_CODE =", result.returncode)

assert result.returncode == 0, (
    "PySCIPOpt 安装失败；停止，不自动更换版本或源码构建"
)

print("\n========== PROTECTED PACKAGE CHECK ==========")

for name, expected in before.items():
    actual = metadata.version(name)
    assert actual == expected, (
        f"受保护依赖发生变化：{name}: {expected} -> {actual}"
    )
    print("UNCHANGED", name, actual)

importlib.invalidate_caches()

print("\n========== IMPORT CHECK ==========")

pyscipopt = importlib.import_module("pyscipopt")
ilpy = importlib.import_module("ilpy")

assert metadata.version("pyscipopt") == "6.2.1"

print("PYSCIPOPT_VERSION =", metadata.version("pyscipopt"))
print("PYSCIPOPT_PATH =", pyscipopt.__file__)
print("ILPY_VERSION =", metadata.version("ilpy"))
print("ILPY_PATH =", ilpy.__file__)

print("\n14.15A8_R7_PYSCIPOPT_IMPORT_PASS")
print("PROTECTED_PACKAGE_VERSIONS=UNCHANGED")
print("GT_READ=NO")
print("INFERENCE_RUN=NO")
print("SOLVER_RUN=NO")

In [ ]:

# ============================================================
# Notebook 14 / 14.15A-8
# Kaggle full import smoke test
#
# Purpose:
#   Verify the mounted frozen source bundle, runtime dependencies,
#   official IO, Warm2 predictor and frozen_v9 policy imports.
#
# No inference, no GT, no evaluation, no output artifacts.
# ============================================================

import sys
import importlib
from pathlib import Path
from importlib import metadata

# ------------------------------------------------------------
# 1. Kaggle environment and source mount
# ------------------------------------------------------------

assert sys.platform == "linux", "This cell must run on Kaggle Linux."
assert Path("/kaggle/working").is_dir(), "Not a Kaggle runtime."

ROOT = Path(
    "/kaggle/input/datasets/markdd520/"
    "biohub-frozen-v9-warm2-bundle/bundle"
)

assert ROOT.is_dir(), f"Source bundle is not mounted: {ROOT}"
assert (ROOT / "bundle_manifest.json").is_file(), (
    "bundle_manifest.json is missing."
)

# Avoid creating __pycache__ files during imports.
sys.dont_write_bytecode = True

# ------------------------------------------------------------
# 2. Source path injection
# ------------------------------------------------------------

paths = [
    ROOT / "external/official/scripts",
    ROOT / "external/official/src",
    ROOT / "src",
    ROOT / "scripts",
]

# Remove existing occurrences, then put paths at the front
# in the exact declared order.
for path in paths:
    assert path.is_dir(), f"Required source directory missing: {path}"

for path in reversed(paths):
    value = str(path)

    while value in sys.path:
        sys.path.remove(value)

    sys.path.insert(0, value)

print("========== 14.15A-8 / PATHS ==========")

for path in paths:
    print("PATH_OK =", path.is_dir(), path)

# ------------------------------------------------------------
# 3. Installed package versions
# ------------------------------------------------------------

print("\n========== PACKAGE VERSIONS ==========")

packages = [
    "torch",
    "numpy",
    "pandas",
    "polars",
    "polars-runtime-32",
    "zarr",
    "numcodecs",
    "donfig",
    "geff",
    "geff-spec",
    "tracksdata",
    "bidict",
    "ilpy",
    "pyscipopt",
    "rustworkx",
    "imagecodecs",
]

for name in packages:
    try:
        version = metadata.version(name)
    except metadata.PackageNotFoundError:
        version = "NOT_INSTALLED"

    print(f"{name} = {version}")

# ------------------------------------------------------------
# 4. Full module import smoke test
# ------------------------------------------------------------

print("\n========== IMPORT SMOKE TEST ==========")

modules = [
    "torch",
    "numpy",
    "pandas",
    "zarr",
    "numcodecs",
    "geff",
    "tracksdata",
    "tracking_cellmot.io",
    "tracking_cellmot.metrics",
    "biohub_cell_tracking.frozen_v9",
    "predict_unet_transformer_mps",
    "train_unet_transformer_mps",
    "evaluate",
    "dataspec",
]

project_modules = {
    "tracking_cellmot.io",
    "tracking_cellmot.metrics",
    "biohub_cell_tracking.frozen_v9",
    "predict_unet_transformer_mps",
    "train_unet_transformer_mps",
    "evaluate",
    "dataspec",
}

loaded = {}

for name in modules:
    module = importlib.import_module(name)

    origin = getattr(module, "__file__", None)
    loaded[name] = origin or "BUILTIN"

    if name in project_modules:
        assert origin is not None, (
            f"Project module has no source path: {name}"
        )

        assert Path(origin).resolve().is_relative_to(ROOT.resolve()), (
            f"Project module resolved outside frozen bundle: "
            f"{name} -> {origin}"
        )

    print("PASS", name, "->", loaded[name], flush=True)

# ------------------------------------------------------------
# 5. Verify required runtime entry points
# ------------------------------------------------------------

print("\n========== KEY OBJECTS ==========")

from biohub_cell_tracking.frozen_v9 import (
    build_frozen_v9_policies,
    apply_frozen_v9,
)

from tracking_cellmot.io import (
    open_dataset,
    save_graph,
)

from predict_unet_transformer_mps import (
    PredictConfig,
    load_model,
)

assert callable(build_frozen_v9_policies)
print("PASS build_frozen_v9_policies =", True)

assert callable(apply_frozen_v9)
print("PASS apply_frozen_v9 =", True)

assert callable(open_dataset)
print("PASS open_dataset =", True)

assert callable(save_graph)
print("PASS save_graph =", True)

assert PredictConfig is not None
print("PASS PredictConfig =", PredictConfig)

assert callable(load_model)
print("PASS load_model =", True)

# ------------------------------------------------------------
# 6. Final status
# ------------------------------------------------------------

print("\n========== 14.15A-8 RESULT ==========")

print("IMPORT_SMOKE_TEST=PASS")
print("GT_READ=NO")
print("INFERENCE_RUN=NO")
print("EVALUATOR_RUN=NO")
print("FILES_WRITTEN=0")

In [ ]:

# Notebook 14 / 14.15B-0
# Read-only predictor entrypoint and output-path audit.

import sys
import re
import ast
from pathlib import Path

assert sys.platform == "linux"
assert Path("/kaggle/working").is_dir()

ROOT = Path(
    "/kaggle/input/datasets/markdd520/"
    "biohub-frozen-v9-warm2-bundle/bundle"
)

PREDICTOR = ROOT / "scripts/predict_unet_transformer_mps.py"

assert ROOT.is_dir()
assert PREDICTOR.is_file()

source = PREDICTOR.read_text(encoding="utf-8")
lines = source.splitlines()
tree = ast.parse(source)

print("========== SOURCE ==========")
print("PREDICTOR =", PREDICTOR)
print("SOURCE_LINES =", len(lines))
print("ROOT_IS_READ_ONLY_INPUT =", str(ROOT).startswith("/kaggle/input/"))

print("\n========== TOP-LEVEL DEFINITIONS ==========")

for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
        print(
            type(node).__name__,
            node.name,
            "LINES",
            node.lineno,
            "-",
            node.end_lineno,
        )

print("\n========== PATH / ARGUMENT / OUTPUT REFERENCES ==========")

pattern = re.compile(
    r"\bROOT\b|PREDICTIONS_PATH|checkpoint|test_splits|"
    r"argparse|ArgumentParser|add_argument|__main__|"
    r"output|mkdir|rmtree|unlink|write_|save|"
    r"CUDA|cuda|mps|device|pool_kernel",
    flags=re.IGNORECASE,
)

matched = [
    number
    for number, line in enumerate(lines, start=1)
    if pattern.search(line)
]

# Print a small amount of surrounding context, merging overlaps.
selected = set()

for number in matched:
    for nearby in range(
        max(1, number - 2),
        min(len(lines), number + 3) + 1,
    ):
        selected.add(nearby)

print("MATCHED_LINES =", len(matched))
print("CONTEXT_LINES =", len(selected))

if len(selected) > 350:
    print("CONTEXT_TRUNCATED_TO_FIRST_350_LINES")
    selected = set(sorted(selected)[:350])

previous = None

for number in sorted(selected):
    if previous is not None and number > previous + 1:
        print("    ...")

    print(f"{number:04d} | {lines[number - 1]}")
    previous = number

print("\n========== SCRIPT END ==========")

start = max(1, len(lines) - 49)

for number in range(start, len(lines) + 1):
    print(f"{number:04d} | {lines[number - 1]}")

print("\n========== RELEVANT BUNDLE FILES ==========")

for path in sorted(ROOT.rglob("*")):
    if not path.is_file():
        continue

    relative = path.relative_to(ROOT)
    name = path.name.lower()

    if (
        "checkpoint" in name
        or "weight" in name
        or name.endswith((".pt", ".pth", ".ckpt", ".safetensors"))
        or "split" in name
    ):
        print(relative, "BYTES =", path.stat().st_size)

print("\n14.15B0_READ_ONLY_AUDIT_COMPLETE")
print("INFERENCE_RUN=NO")
print("GT_READ=NO")
print("FILES_WRITTEN=0")

In [ ]:

# Notebook 14 / 14.15B-0-R1
# Focused, read-only predictor interface audit.

import sys
import json
import inspect
from pathlib import Path

assert sys.platform == "linux"
assert Path("/kaggle/working").is_dir()

ROOT = Path(
    "/kaggle/input/datasets/markdd520/"
    "biohub-frozen-v9-warm2-bundle/bundle"
)

SCRIPT = ROOT / "scripts/predict_unet_transformer_mps.py"

assert SCRIPT.is_file()
sys.dont_write_bytecode = True

lines = SCRIPT.read_text(encoding="utf-8").splitlines()

def show_lines(title, first, last):
    print(f"\n========== {title} ==========")
    for line_no in range(first, min(last, len(lines)) + 1):
        print(f"{line_no:04d} | {lines[line_no - 1]}")

# Exact source imports / path definitions.
show_lines("IMPORTS AND PATH DEFINITIONS", 1, 65)

# Config fields needed to preserve the frozen prediction behavior.
show_lines("PREDICT CONFIG", 66, 112)

# Test-list resolution, output cleanup, graph writing, evaluation gate.
show_lines("PREDICT FUNCTION", 523, 618)

print("\n========== BUNDLED CHECKPOINT CONFIG ==========")

WEIGHTS = (
    ROOT
    / "models/official_baseline/"
      "official_baseline_fold0_warm2_lr1e5/"
      "split_0/edge_predictor_best.pth"
)

assert WEIGHTS.is_file()

CONFIG = WEIGHTS.parent / "config.json"

print("WEIGHTS_EXISTS =", WEIGHTS.is_file())
print("WEIGHTS_BYTES =", WEIGHTS.stat().st_size)
print("CONFIG_EXISTS =", CONFIG.is_file())

if CONFIG.is_file():
    print(
        "CHECKPOINT_CONFIG =",
        json.dumps(
            json.loads(CONFIG.read_text(encoding="utf-8")),
            ensure_ascii=False,
            indent=2,
        ),
    )

print("\n========== PROJECT PATH DEFINITIONS ==========")

TRAIN = ROOT / "scripts/train_unet_transformer_mps.py"

assert TRAIN.is_file()

train_lines = TRAIN.read_text(encoding="utf-8").splitlines()

for number, line in enumerate(train_lines, 1):
    if (
        "PROJECT_ROOT" in line
        or "WEIGHTS_PATH" in line
        or "USERNAME" in line
    ):
        print(f"{number:04d} | {line}")

print("\n========== STATUS ==========")
print("14.15B0_R1_INTERFACE_AUDIT_COMPLETE")
print("FROZEN_SOURCE_MODIFIED=NO")
print("FILES_WRITTEN=0")
print("INFERENCE_RUN=NO")
print("GT_READ=NO")

In [ ]:

# ============================================================
# Notebook 14 / 14.15B-1
# Kaggle test-input discovery and runtime output-path preflight
# Read-only; no inference, no GT, no file writes.
# ============================================================

import sys
import inspect
from pathlib import Path

ROOT = Path(
    "/kaggle/input/datasets/markdd520/"
    "biohub-frozen-v9-warm2-bundle/bundle"
)

DATA_ROOT = Path(
    "/kaggle/input/competitions/"
    "biohub-cell-tracking-during-development"
)

WORK_ROOT = Path("/kaggle/working/biohub_1415")

assert sys.platform == "linux"
assert ROOT.is_dir()
assert DATA_ROOT.is_dir()
assert Path("/kaggle/working").is_dir()

sys.dont_write_bytecode = True

# Restore the known-good module search order if necessary.
project_paths = [
    ROOT / "external/official/scripts",
    ROOT / "external/official/src",
    ROOT / "src",
    ROOT / "scripts",
]

for path in reversed(project_paths):
    assert path.is_dir(), path
    value = str(path)
    while value in sys.path:
        sys.path.remove(value)
    sys.path.insert(0, value)

import predict_unet_transformer_mps as predictor

print("========== PREDICTOR ==========")
print("MODULE =", predictor.__file__)
print("SIGNATURE =", inspect.signature(predictor.predict))

assert Path(predictor.__file__).resolve().is_relative_to(
    ROOT.resolve()
)

print("\n========== FROZEN CONFIG ==========")

cfg = predictor.PredictConfig(
    det_threshold=0.995,
    pool_kernel_um=3.0,
    use_ilp=False,
)

print("DET_THRESHOLD =", cfg.det_threshold)
print("POOL_KERNEL_UM =", cfg.pool_kernel_um)
print("USE_ILP =", cfg.use_ilp)
print("MAX_PARENTS =", cfg.max_parents_per_node)
print("MAX_CHILDREN =", cfg.max_children_per_node)

assert cfg.det_threshold == 0.995
assert cfg.pool_kernel_um == 3.0
assert cfg.use_ilp is False

print("\n========== CHECKPOINT ==========")

weights = (
    ROOT
    / "models/official_baseline/"
      "official_baseline_fold0_warm2_lr1e5/"
      "split_0/edge_predictor_best.pth"
)

assert weights.is_file()

print("WEIGHTS =", weights)
print("WEIGHTS_BYTES =", weights.stat().st_size)

print("\n========== COMPETITION INPUT STRUCTURE ==========")

print("DATA_ROOT =", DATA_ROOT)

top_level = sorted(DATA_ROOT.iterdir(), key=lambda p: p.name)

for path in top_level:
    kind = "DIR" if path.is_dir() else "FILE"
    print("TOP", kind, path.name)

print("\n========== TEST-RELATED DIRECTORIES ==========")

test_related = [
    path for path in top_level
    if path.is_dir()
    and (
        "test" in path.name.lower()
        or path.name.lower().endswith(".zarr")
    )
]

for parent in test_related:
    print("PARENT =", parent)

    children = sorted(parent.iterdir(), key=lambda p: p.name)

    print("CHILD_COUNT =", len(children))

    for child in children[:40]:
        kind = "DIR" if child.is_dir() else "FILE"
        print(" ", kind, child.name)

    if len(children) > 40:
        print(" ...", len(children) - 40, "additional entries")

print("\n========== OUTPUT REDIRECTION FEASIBILITY ==========")

original_output = predictor.PREDICTIONS_PATH
candidate_output = WORK_ROOT / "predictions"

assert candidate_output.resolve().is_relative_to(
    Path("/kaggle/working").resolve()
)

try:
    # Assignment only. Do not invoke predict().
    predictor.PREDICTIONS_PATH = candidate_output

    print("ORIGINAL_OUTPUT =", original_output)
    print("CANDIDATE_OUTPUT =", predictor.PREDICTIONS_PATH)

    assert predictor.PREDICTIONS_PATH == candidate_output

finally:
    # Restore the module's original value after the dry check.
    predictor.PREDICTIONS_PATH = original_output

print("OUTPUT_RESTORED =", predictor.PREDICTIONS_PATH == original_output)

print("\n14.15B1_PREFLIGHT_COMPLETE")
print("INFERENCE_RUN=NO")
print("GT_READ=NO")
print("FILES_WRITTEN=0")

In [ ]:

# Notebook 14 / 14.15B-2
# Prepare a dynamic test split. No inference.

import json
import sys
from pathlib import Path

assert sys.platform == "linux"

DATA_DIR = Path(
    "/kaggle/input/competitions/"
    "biohub-cell-tracking-during-development/test"
)
WORK_ROOT = Path("/kaggle/working/biohub_1415")
SPLITS = WORK_ROOT / "test_splits_1415.json"

USERNAME = "kaggle_1415"
METHOD = "warm2_det0995_v1"

OUTPUT_DIR = (
    WORK_ROOT / "predictions"
    / USERNAME / METHOD / "split_0"
)

assert DATA_DIR.is_dir()
assert Path("/kaggle/working").is_dir()
assert OUTPUT_DIR.resolve().is_relative_to(
    Path("/kaggle/working").resolve()
)

# The predictor deletes existing *.geff in its selected output directory.
# Never allow this run to overwrite or clean up previous results.
assert not OUTPUT_DIR.exists(), (
    f"输出目录已存在，禁止覆盖：{OUTPUT_DIR}"
)

assert not SPLITS.exists(), (
    f"测试列表已存在；不要重复执行准备 Cell：{SPLITS}"
)

test_names = sorted(
    p.name
    for p in DATA_DIR.iterdir()
    if p.is_dir() and p.name.endswith(".zarr")
)

assert test_names, "未发现测试 Zarr"
assert len(test_names) == len(set(test_names))

# predict() reads folds[fold]["test"], with fold=0.
payload = [{"test": test_names}]

WORK_ROOT.mkdir(parents=True, exist_ok=True)

with SPLITS.open("x", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)
    f.write("\n")

assert json.loads(SPLITS.read_text(encoding="utf-8")) == payload

print("========== 14.15B-2 ==========")
print("DATA_DIR =", DATA_DIR)
print("TEST_COUNT =", len(test_names))

for name in test_names:
    print("TEST =", name)

print("SPLITS =", SPLITS)
print("OUTPUT_DIR =", OUTPUT_DIR)
print("OUTPUT_EXISTS =", OUTPUT_DIR.exists())

print("\n14.15B2_DYNAMIC_TEST_SPLIT_READY")
print("GT_READ=NO")
print("INFERENCE_RUN=NO")

In [ ]:

# Notebook 14 / 14.15B-3
# Full Kaggle CUDA inference: Warm2 / det_threshold=0.995
# No ground truth and no evaluator.

import sys
import json
import hashlib
from pathlib import Path

import torch

ROOT = Path(
    "/kaggle/input/datasets/markdd520/"
    "biohub-frozen-v9-warm2-bundle/bundle"
)

DATA_DIR = Path(
    "/kaggle/input/competitions/"
    "biohub-cell-tracking-during-development/test"
)

WORK_ROOT = Path("/kaggle/working/biohub_1415")
SPLITS = WORK_ROOT / "test_splits_1415.json"

USERNAME = "kaggle_1415"
METHOD = "warm2_det0995_v1"

PREDICTIONS_ROOT = WORK_ROOT / "predictions"

OUTPUT_DIR = (
    PREDICTIONS_ROOT / USERNAME / METHOD / "split_0"
)

WEIGHTS = (
    ROOT
    / "models/official_baseline/"
      "official_baseline_fold0_warm2_lr1e5/"
      "split_0/edge_predictor_best.pth"
)

assert ROOT.is_dir()
assert DATA_DIR.is_dir()
assert SPLITS.is_file()
assert WEIGHTS.is_file()
assert OUTPUT_DIR.resolve().is_relative_to(
    Path("/kaggle/working").resolve()
)

assert not OUTPUT_DIR.exists(), (
    "输出目录已存在，禁止重跑或覆盖。"
)

assert torch.cuda.is_available(), (
    "CUDA 不可用：停止，不允许静默回退到 CPU。"
)

test_names = json.loads(
    SPLITS.read_text(encoding="utf-8")
)[0]["test"]

actual_names = sorted(
    p.name
    for p in DATA_DIR.iterdir()
    if p.is_dir() and p.name.endswith(".zarr")
)

assert test_names == actual_names, (
    "测试列表与当前挂载的测试数据不一致。"
)

sys.dont_write_bytecode = True

paths = [
    ROOT / "external/official/scripts",
    ROOT / "external/official/src",
    ROOT / "src",
    ROOT / "scripts",
]

for path in reversed(paths):
    assert path.is_dir(), path
    value = str(path)
    while value in sys.path:
        sys.path.remove(value)
    sys.path.insert(0, value)

import predict_unet_transformer_mps as predictor

assert Path(predictor.__file__).resolve().is_relative_to(
    ROOT.resolve()
)

cfg = predictor.PredictConfig(
    det_threshold=0.995,
    pool_kernel_um=3.0,
    use_ilp=False,
)

assert cfg.det_threshold == 0.995
assert cfg.pool_kernel_um == 3.0
assert cfg.det_tta is True
assert cfg.edge_activation == "softmax"
assert cfg.use_ilp is False
assert cfg.max_parents_per_node == 1
assert cfg.max_children_per_node == 2

def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("========== 14.15B-3 / PRE-INFERENCE ==========", flush=True)
print("CUDA_DEVICE =", torch.cuda.get_device_name(0), flush=True)
print("WEIGHTS_SHA256 =", sha256_file(WEIGHTS), flush=True)
print("TEST_COUNT =", len(test_names), flush=True)
print("DET_THRESHOLD =", cfg.det_threshold, flush=True)
print("POOL_KERNEL_UM =", cfg.pool_kernel_um, flush=True)
print("USE_ILP =", cfg.use_ilp, flush=True)
print("OUTPUT_DIR =", OUTPUT_DIR, flush=True)
print("EVALUATE = False", flush=True)
print("MAX_FRAMES = None", flush=True)

# Redirect only runtime output-related module globals.
# The source file and bundled weights remain unchanged.
original_predictions_path = predictor.PREDICTIONS_PATH
original_username = predictor.USERNAME

try:
    predictor.PREDICTIONS_PATH = PREDICTIONS_ROOT
    predictor.USERNAME = USERNAME

    assert (
        predictor.PREDICTIONS_PATH
        / predictor.USERNAME
        / METHOD
        / "split_0"
    ) == OUTPUT_DIR

    predictor.predict(
        data_dir=DATA_DIR,
        fold=0,
        splits_file=SPLITS,
        weights_path=WEIGHTS,
        cfg=cfg,
        method=METHOD,
        debug_video=None,
        unet_batch_size=4,
        video_slice=None,
        evaluate=False,
        max_frames=None,
    )

finally:
    predictor.PREDICTIONS_PATH = original_predictions_path
    predictor.USERNAME = original_username

print("\n========== OUTPUT CHECK ==========", flush=True)

assert OUTPUT_DIR.is_dir(), "预测输出目录不存在"

actual_geffs = sorted(
    p.name for p in OUTPUT_DIR.glob("*.geff")
)

expected_geffs = sorted(
    f"{name}.geff" for name in test_names
)

print("EXPECTED_GEFFS =", expected_geffs)
print("ACTUAL_GEFFS =", actual_geffs)

assert actual_geffs == expected_geffs, (
    "输出 GEFF 列表与测试序列不一致"
)

print("\n14.15B3_CUDA_INFERENCE_OUTPUTS_PRESENT")
print("GT_READ=NO")
print("EVALUATOR_RUN=NO")
print("FROZEN_SOURCE_MODIFIED=NO")

In [ ]:

# ============================================================
# Notebook 14 / 14.15B-4
# Read-only inventory of det=0.995 GEFF outputs.
#
# Does not read GT, evaluate, infer, or modify predictions.
# ============================================================

import json
import sys
from pathlib import Path

import zarr

assert sys.platform == "linux"

WORK_ROOT = Path("/kaggle/working/biohub_1415")
SPLITS = WORK_ROOT / "test_splits_1415.json"

OUTPUT_DIR = (
    WORK_ROOT / "predictions"
    / "kaggle_1415"
    / "warm2_det0995_v1"
    / "split_0"
)

assert SPLITS.is_file()
assert OUTPUT_DIR.is_dir()
assert OUTPUT_DIR.resolve().is_relative_to(
    Path("/kaggle/working").resolve()
)

test_names = json.loads(
    SPLITS.read_text(encoding="utf-8")
)[0]["test"]

expected_names = sorted(
    f"{name}.geff" for name in test_names
)

actual_names = sorted(
    p.name for p in OUTPUT_DIR.glob("*.geff")
)

assert actual_names == expected_names, (
    "GEFF 输出与测试列表不一致"
)

def inspect_group(group, prefix="", depth=0, max_depth=5):
    """Inspect Zarr metadata only; do not materialize array contents."""
    if depth > max_depth:
        print("  DEPTH_LIMIT:", prefix)
        return 0

    array_count = 0

    for key in sorted(group.array_keys()):
        arr = group[key]
        name = f"{prefix}/{key}" if prefix else key

        print(
            "  ARRAY",
            name,
            "SHAPE =", arr.shape,
            "DTYPE =", arr.dtype,
        )
        array_count += 1

    for key in sorted(group.group_keys()):
        child = group[key]
        name = f"{prefix}/{key}" if prefix else key

        print("  GROUP", name)

        array_count += inspect_group(
            child,
            prefix=name,
            depth=depth + 1,
            max_depth=max_depth,
        )

    return array_count

print("========== 14.15B-4 / READ-ONLY GEFF INSPECTION ==========")

total_arrays = 0

for name in expected_names:
    path = OUTPUT_DIR / name

    assert path.is_dir(), (
        f"GEFF 不是预期的目录结构：{path}"
    )

    file_count = sum(
        p.is_file() for p in path.rglob("*")
    )

    total_bytes = sum(
        p.stat().st_size
        for p in path.rglob("*")
        if p.is_file()
    )

    assert file_count > 0
    assert total_bytes > 0

    print(f"\n========== {name} ==========")
    print("PATH =", path)
    print("FILE_COUNT =", file_count)
    print("TOTAL_BYTES =", total_bytes)

    # Open in read-only mode. Do not access or modify the arrays.
    group = zarr.open_group(
        store=str(path),
        mode="r",
    )

    print("ZARR_FORMAT =", group.metadata.zarr_format)
    print("ROOT_ATTR_KEYS =", sorted(group.attrs.keys()))

    array_count = inspect_group(group)

    print("ARRAY_COUNT =", array_count)

    assert array_count > 0, (
        f"GEFF 中未找到可识别的 Zarr 数组：{name}"
    )

    total_arrays += array_count

print("\n========== 14.15B-4 RESULT ==========")
print("GEFF_COUNT =", len(expected_names))
print("TOTAL_ARRAYS =", total_arrays)

print("14.15B4_GEFF_METADATA_READABLE=PASS")
print("GEFF_SEMANTICS_VERIFIED=NO")
print("GT_READ=NO")
print("EVALUATOR_RUN=NO")
print("INFERENCE_RUN=NO")
print("PREDICTIONS_MODIFIED=NO")

In [ ]:

# ============================================================
# Notebook 14 / 14.15B-5
# Read-only semantic validation of det=0.995 GEFF predictions.
#
# Check:
#   - unique node IDs
#   - valid node coordinates and timestamps
#   - all edge endpoints reference existing nodes
#   - edges connect consecutive frames in forward time
#   - finite, bounded edge probabilities
#   - finite, nonnegative edge distances
#
# No GT, no evaluator, no inference, no file writes.
# ============================================================

import json
import sys
from pathlib import Path

import numpy as np
import zarr

assert sys.platform == "linux"

WORK_ROOT = Path("/kaggle/working/biohub_1415")
SPLITS = WORK_ROOT / "test_splits_1415.json"

OUTPUT_DIR = (
    WORK_ROOT
    / "predictions"
    / "kaggle_1415"
    / "warm2_det0995_v1"
    / "split_0"
)

assert SPLITS.is_file()
assert OUTPUT_DIR.is_dir()

assert OUTPUT_DIR.resolve().is_relative_to(
    Path("/kaggle/working").resolve()
)

test_names = json.loads(
    SPLITS.read_text(encoding="utf-8")
)[0]["test"]

expected_geffs = sorted(
    f"{name}.geff" for name in test_names
)

actual_geffs = sorted(
    p.name for p in OUTPUT_DIR.glob("*.geff")
)

assert actual_geffs == expected_geffs, (
    "GEFF 列表与当前动态测试列表不一致"
)

print("========== 14.15B-5 / GEFF SEMANTICS ==========", flush=True)

total_nodes = 0
total_edges = 0

for geff_name in expected_geffs:
    path = OUTPUT_DIR / geff_name

    # Explicitly read-only.
    group = zarr.open_group(
        store=str(path),
        mode="r",
    )

    node_ids = np.asarray(group["nodes/ids"][:])
    times = np.asarray(group["nodes/props/t/values"][:])

    x = np.asarray(group["nodes/props/x/values"][:])
    y = np.asarray(group["nodes/props/y/values"][:])
    z = np.asarray(group["nodes/props/z/values"][:])

    edge_ids = np.asarray(group["edges/ids"][:])
    edge_dist = np.asarray(
        group["edges/props/edge_dist/values"][:]
    )
    edge_prob = np.asarray(
        group["edges/props/edge_prob/values"][:]
    )

    n = len(node_ids)
    e = len(edge_ids)

    print(f"\n========== {geff_name} ==========", flush=True)
    print("NODES =", n, "EDGES =", e, flush=True)

    # --------------------------------------------------------
    # Node structure
    # --------------------------------------------------------

    assert node_ids.ndim == 1
    assert times.shape == (n,)

    assert x.shape == (n,)
    assert y.shape == (n,)
    assert z.shape == (n,)

    assert n > 0, "GEFF 包含零个节点"

    order = np.argsort(node_ids)
    sorted_ids = node_ids[order]

    assert np.all(sorted_ids[1:] != sorted_ids[:-1]), (
        "检测到重复 node ID"
    )

    assert np.all(times >= 0), "检测到负时间戳"

    for axis, values in (("x", x), ("y", y), ("z", z)):
        assert np.all(np.isfinite(values)), (
            f"{axis} 包含 NaN 或 Inf"
        )

    print(
        "TIME_RANGE =",
        int(times.min()),
        int(times.max()),
    )
    print("UNIQUE_NODE_IDS = PASS")
    print("FINITE_COORDINATES = PASS")

    # --------------------------------------------------------
    # Edge structure and attributes
    # --------------------------------------------------------

    assert edge_ids.shape == (e, 2)
    assert edge_dist.shape == (e,)
    assert edge_prob.shape == (e,)

    assert np.all(np.isfinite(edge_dist)), (
        "edge_dist 包含 NaN 或 Inf"
    )
    assert np.all(edge_dist >= 0), (
        "edge_dist 包含负值"
    )

    assert np.all(np.isfinite(edge_prob)), (
        "edge_prob 包含 NaN 或 Inf"
    )

    assert np.all(
        (edge_prob >= 0.0) & (edge_prob <= 1.0)
    ), "edge_prob 超出 [0, 1]"

    if e > 0:
        # Resolve endpoint IDs without assuming IDs are contiguous
        # or that they match node-array positions.
        flat_endpoints = edge_ids.reshape(-1)

        locations = np.searchsorted(
            sorted_ids,
            flat_endpoints,
        )

        in_range = locations < n

        assert np.all(in_range), (
            "某些 edge endpoint ID 不在节点 ID 范围内"
        )

        assert np.array_equal(
            sorted_ids[locations],
            flat_endpoints,
        ), "某些 edge endpoint ID 未在节点列表中找到"

        endpoint_rows = order[locations].reshape(e, 2)

        source_times = times[endpoint_rows[:, 0]]
        target_times = times[endpoint_rows[:, 1]]

        dt = target_times - source_times

        unique_dt, dt_counts = np.unique(
            dt,
            return_counts=True,
        )

        print(
            "EDGE_DT_DISTRIBUTION =",
            {
                int(delta): int(count)
                for delta, count in zip(unique_dt, dt_counts)
            },
        )

        assert np.all(dt == 1), (
            "预测边未全部连接相邻帧，或边方向错误"
        )

        print(
            "EDGE_PROB_RANGE =",
            float(edge_prob.min()),
            float(edge_prob.max()),
        )

        print(
            "EDGE_DIST_RANGE =",
            float(edge_dist.min()),
            float(edge_dist.max()),
        )

    print("VALID_EDGE_ENDPOINTS = PASS")
    print("FORWARD_CONSECUTIVE_EDGES = PASS")
    print("FINITE_EDGE_ATTRIBUTES = PASS")

    total_nodes += n
    total_edges += e

print("\n========== 14.15B-5 RESULT ==========")
print("GEFF_COUNT =", len(expected_geffs))
print("TOTAL_NODES =", total_nodes)
print("TOTAL_EDGES =", total_edges)

print("14.15B5_GEFF_SEMANTICS_PASS")
print("GT_READ=NO")
print("EVALUATOR_RUN=NO")
print("INFERENCE_RUN=NO")
print("PREDICTIONS_MODIFIED=NO")

In [ ]:

# ============================================================
# Notebook 14 / 14.15B-6
# Full Kaggle CUDA inference: Warm2 / det_threshold=0.9975
#
# Preserves existing det=0.995 predictions.
# No GT, no evaluator, no ILP, no frame limit.
# ============================================================

import sys
import json
import hashlib
from pathlib import Path

import torch

ROOT = Path(
    "/kaggle/input/datasets/markdd520/"
    "biohub-frozen-v9-warm2-bundle/bundle"
)

DATA_DIR = Path(
    "/kaggle/input/competitions/"
    "biohub-cell-tracking-during-development/test"
)

WORK_ROOT = Path("/kaggle/working/biohub_1415")
SPLITS = WORK_ROOT / "test_splits_1415.json"

USERNAME = "kaggle_1415"
METHOD = "warm2_det09975_v1"

PREDICTIONS_ROOT = WORK_ROOT / "predictions"

OUTPUT_DIR = (
    PREDICTIONS_ROOT
    / USERNAME
    / METHOD
    / "split_0"
)

PREVIOUS_OUTPUT_DIR = (
    PREDICTIONS_ROOT
    / USERNAME
    / "warm2_det0995_v1"
    / "split_0"
)

WEIGHTS = (
    ROOT
    / "models/official_baseline/"
      "official_baseline_fold0_warm2_lr1e5/"
      "split_0/edge_predictor_best.pth"
)

EXPECTED_WEIGHTS_SHA = (
    "31d209bfde1fd5bbf3b2a661cdb4a25"
    "bb72922b832d315ee5a75cc32f44656c1"
)

def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(
            lambda: stream.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

# ------------------------------------------------------------
# 1. Source, data, checkpoint and output safety
# ------------------------------------------------------------

assert sys.platform == "linux"
assert Path("/kaggle/working").is_dir()

assert ROOT.is_dir()
assert DATA_DIR.is_dir()
assert SPLITS.is_file()
assert WEIGHTS.is_file()

assert sha256_file(WEIGHTS) == EXPECTED_WEIGHTS_SHA, (
    "Checkpoint SHA 不匹配，停止"
)

assert OUTPUT_DIR.resolve().is_relative_to(
    Path("/kaggle/working").resolve()
)

assert OUTPUT_DIR != PREVIOUS_OUTPUT_DIR

# predict() removes existing *.geff in its selected directory.
# Never rerun into an existing output directory.
assert not OUTPUT_DIR.exists(), (
    f"新输出目录已存在，禁止覆盖：{OUTPUT_DIR}"
)

assert PREVIOUS_OUTPUT_DIR.is_dir(), (
    "此前的 0.995 输出目录不存在"
)

test_names = json.loads(
    SPLITS.read_text(encoding="utf-8")
)[0]["test"]

actual_names = sorted(
    p.name
    for p in DATA_DIR.iterdir()
    if p.is_dir() and p.name.endswith(".zarr")
)

assert test_names == actual_names, (
    "动态测试列表与当前数据不一致"
)

expected_geffs = sorted(
    f"{name}.geff" for name in test_names
)

previous_geffs = sorted(
    p.name
    for p in PREVIOUS_OUTPUT_DIR.glob("*.geff")
)

assert previous_geffs == expected_geffs, (
    "此前的 0.995 GEFF 列表不完整"
)

assert torch.cuda.is_available(), (
    "CUDA 不可用，禁止静默回退 CPU"
)

# ------------------------------------------------------------
# 2. Frozen module import
# ------------------------------------------------------------

sys.dont_write_bytecode = True

paths = [
    ROOT / "external/official/scripts",
    ROOT / "external/official/src",
    ROOT / "src",
    ROOT / "scripts",
]

for path in reversed(paths):
    assert path.is_dir(), path
    value = str(path)

    while value in sys.path:
        sys.path.remove(value)

    sys.path.insert(0, value)

import predict_unet_transformer_mps as predictor

assert Path(predictor.__file__).resolve().is_relative_to(
    ROOT.resolve()
)

cfg = predictor.PredictConfig(
    det_threshold=0.9975,
    pool_kernel_um=3.0,
    use_ilp=False,
)

assert cfg.det_threshold == 0.9975
assert cfg.pool_kernel_um == 3.0
assert cfg.det_tta is True
assert cfg.edge_activation == "softmax"
assert cfg.use_ilp is False
assert cfg.max_parents_per_node == 1
assert cfg.max_children_per_node == 2

# ------------------------------------------------------------
# 3. CUDA inference
# ------------------------------------------------------------

print(
    "========== 14.15B-6 / PRE-INFERENCE ==========",
    flush=True,
)

print("CUDA_DEVICE =", torch.cuda.get_device_name(0))
print("WEIGHTS_SHA256 =", sha256_file(WEIGHTS))
print("TEST_COUNT =", len(test_names))
print("DET_THRESHOLD =", cfg.det_threshold)
print("POOL_KERNEL_UM =", cfg.pool_kernel_um)
print("USE_ILP =", cfg.use_ilp)
print("OUTPUT_DIR =", OUTPUT_DIR)
print("PREVIOUS_OUTPUT_PRESERVED =", PREVIOUS_OUTPUT_DIR)
print("EVALUATE = False")
print("MAX_FRAMES = None")

original_predictions_path = predictor.PREDICTIONS_PATH
original_username = predictor.USERNAME

try:
    predictor.PREDICTIONS_PATH = PREDICTIONS_ROOT
    predictor.USERNAME = USERNAME

    assert (
        predictor.PREDICTIONS_PATH
        / predictor.USERNAME
        / METHOD
        / "split_0"
    ) == OUTPUT_DIR

    predictor.predict(
        data_dir=DATA_DIR,
        fold=0,
        splits_file=SPLITS,
        weights_path=WEIGHTS,
        cfg=cfg,
        method=METHOD,
        debug_video=None,
        unet_batch_size=4,
        video_slice=None,
        evaluate=False,
        max_frames=None,
    )

finally:
    predictor.PREDICTIONS_PATH = original_predictions_path
    predictor.USERNAME = original_username

# ------------------------------------------------------------
# 4. Output inventory
# ------------------------------------------------------------

print("\n========== OUTPUT CHECK ==========", flush=True)

assert OUTPUT_DIR.is_dir(), "0.9975 输出目录不存在"

actual_geffs = sorted(
    p.name
    for p in OUTPUT_DIR.glob("*.geff")
)

previous_geffs_after = sorted(
    p.name
    for p in PREVIOUS_OUTPUT_DIR.glob("*.geff")
)

print("EXPECTED_GEFFS =", expected_geffs)
print("ACTUAL_GEFFS =", actual_geffs)

assert actual_geffs == expected_geffs, (
    "0.9975 GEFF 列表不完整"
)

assert previous_geffs_after == previous_geffs, (
    "此前的 0.995 GEFF 列表发生变化"
)

print("\n14.15B6_CUDA_INFERENCE_OUTPUTS_PRESENT")
print("PREVIOUS_GEFF_LIST_PRESERVED=YES")
print("GEFF_SEMANTICS_VERIFIED=NO")
print("GT_READ=NO")
print("EVALUATOR_RUN=NO")
print("FROZEN_SOURCE_MODIFIED=NO")

In [ ]:

# ============================================================
# Notebook 14 / 14.15B-7
# Read-only semantic validation of det=0.9975 GEFF outputs.
#
# No inference, GT, evaluation, or file modifications.
# ============================================================

import json
import sys
from pathlib import Path

import numpy as np
import zarr

assert sys.platform == "linux"

WORK_ROOT = Path("/kaggle/working/biohub_1415")
SPLITS = WORK_ROOT / "test_splits_1415.json"

OUTPUT_DIR = (
    WORK_ROOT
    / "predictions"
    / "kaggle_1415"
    / "warm2_det09975_v1"
    / "split_0"
)

PREVIOUS_DIR = (
    WORK_ROOT
    / "predictions"
    / "kaggle_1415"
    / "warm2_det0995_v1"
    / "split_0"
)

assert SPLITS.is_file()
assert OUTPUT_DIR.is_dir()
assert PREVIOUS_DIR.is_dir()

assert OUTPUT_DIR.resolve().is_relative_to(
    Path("/kaggle/working").resolve()
)

test_names = json.loads(
    SPLITS.read_text(encoding="utf-8")
)[0]["test"]

expected_geffs = sorted(
    f"{name}.geff" for name in test_names
)

actual_geffs = sorted(
    p.name for p in OUTPUT_DIR.glob("*.geff")
)

previous_geffs = sorted(
    p.name for p in PREVIOUS_DIR.glob("*.geff")
)

assert actual_geffs == expected_geffs, (
    "0.9975 GEFF 列表与测试列表不一致"
)

assert previous_geffs == expected_geffs, (
    "0.995 GEFF 列表不完整"
)

print(
    "========== 14.15B-7 / GEFF SEMANTICS ==========",
    flush=True,
)

total_nodes = 0
total_edges = 0

for geff_name in expected_geffs:
    path = OUTPUT_DIR / geff_name

    # Read-only Zarr access.
    group = zarr.open_group(
        store=str(path),
        mode="r",
    )

    node_ids = np.asarray(group["nodes/ids"][:])
    times = np.asarray(
        group["nodes/props/t/values"][:]
    )

    x = np.asarray(group["nodes/props/x/values"][:])
    y = np.asarray(group["nodes/props/y/values"][:])
    z = np.asarray(group["nodes/props/z/values"][:])

    edge_ids = np.asarray(group["edges/ids"][:])

    edge_dist = np.asarray(
        group["edges/props/edge_dist/values"][:]
    )
    edge_prob = np.asarray(
        group["edges/props/edge_prob/values"][:]
    )

    n = len(node_ids)
    e = len(edge_ids)

    print(
        f"\n========== {geff_name} ==========",
        flush=True,
    )
    print("NODES =", n, "EDGES =", e)

    # --------------------------------------------------------
    # Nodes
    # --------------------------------------------------------

    assert node_ids.ndim == 1
    assert n > 0

    assert times.shape == (n,)
    assert x.shape == (n,)
    assert y.shape == (n,)
    assert z.shape == (n,)

    order = np.argsort(node_ids)
    sorted_ids = node_ids[order]

    assert np.all(
        sorted_ids[1:] != sorted_ids[:-1]
    ), "重复 node ID"

    assert np.all(times >= 0), "存在负时间戳"

    for axis, values in (
        ("x", x),
        ("y", y),
        ("z", z),
    ):
        assert np.all(np.isfinite(values)), (
            f"{axis} 包含 NaN 或 Inf"
        )

    print(
        "TIME_RANGE =",
        int(times.min()),
        int(times.max()),
    )
    print("UNIQUE_NODE_IDS = PASS")
    print("FINITE_COORDINATES = PASS")

    # --------------------------------------------------------
    # Edges
    # --------------------------------------------------------

    assert edge_ids.shape == (e, 2)
    assert edge_dist.shape == (e,)
    assert edge_prob.shape == (e,)

    assert np.all(np.isfinite(edge_dist)), (
        "edge_dist 包含 NaN 或 Inf"
    )

    assert np.all(edge_dist >= 0), (
        "edge_dist 包含负值"
    )

    assert np.all(np.isfinite(edge_prob)), (
        "edge_prob 包含 NaN 或 Inf"
    )

    assert np.all(
        (edge_prob >= 0.0) &
        (edge_prob <= 1.0)
    ), "edge_prob 超出 [0, 1]"

    if e > 0:
        endpoints = edge_ids.reshape(-1)

        # Resolve actual node IDs, not assumed row indices.
        locations = np.searchsorted(
            sorted_ids,
            endpoints,
        )

        assert np.all(locations < n), (
            "边端点超出节点 ID 查找范围"
        )

        assert np.array_equal(
            sorted_ids[locations],
            endpoints,
        ), "存在未定义的边端点"

        endpoint_rows = order[
            locations
        ].reshape(e, 2)

        source_times = times[endpoint_rows[:, 0]]
        target_times = times[endpoint_rows[:, 1]]

        dt = target_times - source_times

        unique_dt, counts = np.unique(
            dt,
            return_counts=True,
        )

        print(
            "EDGE_DT_DISTRIBUTION =",
            {
                int(delta): int(count)
                for delta, count in zip(
                    unique_dt,
                    counts,
                )
            },
        )

        assert np.all(dt == 1), (
            "存在非相邻帧边或反向边"
        )

        print(
            "EDGE_PROB_RANGE =",
            float(edge_prob.min()),
            float(edge_prob.max()),
        )
        print(
            "EDGE_DIST_RANGE =",
            float(edge_dist.min()),
            float(edge_dist.max()),
        )

    print("VALID_EDGE_ENDPOINTS = PASS")
    print("FORWARD_CONSECUTIVE_EDGES = PASS")
    print("FINITE_EDGE_ATTRIBUTES = PASS")

    total_nodes += n
    total_edges += e

print("\n========== 14.15B-7 RESULT ==========")

print("GEFF_COUNT =", len(expected_geffs))
print("TOTAL_NODES =", total_nodes)
print("TOTAL_EDGES =", total_edges)

print("14.15B7_GEFF_SEMANTICS_PASS")
print("PREVIOUS_GEFF_LIST_PRESERVED=YES")
print("GT_READ=NO")
print("EVALUATOR_RUN=NO")
print("INFERENCE_RUN=NO")
print("PREDICTIONS_MODIFIED=NO")

In [ ]:

# ============================================================
# Notebook 14 / 14.15C-0
# Frozen-v9 tracking interface and call-site audit.
#
# Read-only source inspection.
# No tracking, GT, evaluator, or file writes.
# ============================================================

import ast
import inspect
import sys
from pathlib import Path

ROOT = Path(
    "/kaggle/input/datasets/markdd520/"
    "biohub-frozen-v9-warm2-bundle/bundle"
)

assert sys.platform == "linux"
assert ROOT.is_dir()
assert Path("/kaggle/working").is_dir()

sys.dont_write_bytecode = True

paths = [
    ROOT / "external/official/scripts",
    ROOT / "external/official/src",
    ROOT / "src",
    ROOT / "scripts",
]

for path in reversed(paths):
    assert path.is_dir(), path
    value = str(path)
    while value in sys.path:
        sys.path.remove(value)
    sys.path.insert(0, value)

import biohub_cell_tracking.frozen_v9 as frozen

source_file = Path(frozen.__file__).resolve()

assert source_file.is_relative_to(ROOT.resolve())

print("========== FROZEN MODULE ==========")
print("SOURCE =", source_file)
print("SOURCE_LINES =", len(source_file.read_text(encoding="utf-8").splitlines()))

print("\n========== PUBLIC CALLABLES ==========")

for name in ("build_frozen_v9_policies", "apply_frozen_v9"):
    obj = getattr(frozen, name)

    assert callable(obj), f"缺少冻结函数：{name}"

    print("\nFUNCTION =", name)
    print("SIGNATURE =", inspect.signature(obj))
    print("SOURCE:")
    print(inspect.getsource(obj))

print("\n========== BUNDLED PYTHON INVENTORY ==========")

py_files = sorted(ROOT.rglob("*.py"))

for file in py_files:
    print(file.relative_to(ROOT))

print("\n========== TRACKING CALL SITES ==========")

needles = (
    "build_frozen_v9_policies",
    "apply_frozen_v9",
    "warm2_det0995",
    "warm2_det09975",
)

total_matches = 0

for file in py_files:
    if file.resolve() == source_file:
        continue

    text = file.read_text(encoding="utf-8")
    lines = text.splitlines()

    hit_lines = [
        i for i, line in enumerate(lines)
        if any(needle in line for needle in needles)
    ]

    if not hit_lines:
        continue

    print("\nFILE =", file.relative_to(ROOT))

    # Print bounded surrounding source for each distinct call site.
    shown = set()

    for hit in hit_lines[:12]:
        first = max(0, hit - 5)
        last = min(len(lines), hit + 9)

        for i in range(first, last):
            if i not in shown:
                print(f"{i + 1:04d} | {lines[i]}")
                shown.add(i)

        print("    ...")

    total_matches += len(hit_lines)

print("\nTOTAL_CALLSITE_MATCHES =", total_matches)

print("\n========== RELEVANT NON-PYTHON FILES ==========")

for file in sorted(ROOT.rglob("*")):
    if not file.is_file():
        continue

    relative = file.relative_to(ROOT)
    name = file.name.lower()

    if (
        "frozen" in name
        or "policy" in name
        or "tracking" in name
        or "submission" in name
    ):
        print(relative, "BYTES =", file.stat().st_size)

print("\n14.15C0_FROZEN_TRACKING_INTERFACE_AUDITED")
print("TRACKING_RUN=NO")
print("GT_READ=NO")
print("EVALUATOR_RUN=NO")
print("FILES_WRITTEN=0")

In [ ]:

# ============================================================
# Notebook 14 / 14.15C-1
# Frozen-v9 data provenance and GEFF writer audit.
#
# Source inspection only:
# - train_dir / prefix_map dependencies
# - prediction GEFF read paths
# - output graph serialization interface
#
# No tracking, GT, evaluation, or file writes.
# ============================================================

import ast
import inspect
import sys
from pathlib import Path

assert sys.platform == "linux"
assert Path("/kaggle/working").is_dir()

ROOT = Path(
    "/kaggle/input/datasets/markdd520/"
    "biohub-frozen-v9-warm2-bundle/bundle"
)

DATA_ROOT = Path(
    "/kaggle/input/competitions/"
    "biohub-cell-tracking-during-development"
)

assert ROOT.is_dir()
assert DATA_ROOT.is_dir()

sys.dont_write_bytecode = True

paths = [
    ROOT / "external/official/scripts",
    ROOT / "external/official/src",
    ROOT / "src",
    ROOT / "scripts",
]

for path in reversed(paths):
    assert path.is_dir(), path
    value = str(path)
    while value in sys.path:
        sys.path.remove(value)
    sys.path.insert(0, value)

import biohub_cell_tracking.frozen_v9 as frozen
from tracking_cellmot.io import save_graph

source_path = Path(frozen.__file__).resolve()
assert source_path.is_relative_to(ROOT.resolve())

source_lines = source_path.read_text(
    encoding="utf-8"
).splitlines()

tree = ast.parse("\n".join(source_lines))

functions = {
    node.name: node
    for node in tree.body
    if isinstance(
        node,
        (ast.FunctionDef, ast.AsyncFunctionDef),
    )
}

def show_function(name):
    print(f"\n========== FUNCTION: {name} ==========")

    if name not in functions:
        print("NOT_FOUND")
        return

    node = functions[name]

    print("LINES =", node.lineno, "-", node.end_lineno)

    for number in range(node.lineno, node.end_lineno + 1):
        print(f"{number:04d} | {source_lines[number - 1]}")

print("========== SOURCE AUTHORITY ==========")
print("FROZEN_SOURCE =", source_path)

# ------------------------------------------------------------
# 1. Exact dataset prefix mapping and scale resolution
# ------------------------------------------------------------

for name in (
    "_prefix_for",
    "census_single_edge_gaps",
    "census_isolated_bridges",
    "prepare_precleanup_graph",
):
    show_function(name)

# ------------------------------------------------------------
# 2. Every train_dir reference in the frozen module
# ------------------------------------------------------------

print("\n========== ALL TRAIN_DIR REFERENCES ==========")

for number, line in enumerate(source_lines, start=1):
    if "train_dir" not in line:
        continue

    print(f"\nREFERENCE_LINE = {number}")

    for nearby in range(
        max(1, number - 3),
        min(len(source_lines), number + 3) + 1,
    ):
        print(
            f"{nearby:04d} | {source_lines[nearby - 1]}"
        )

# ------------------------------------------------------------
# 3. Label / ground-truth access indicators
# ------------------------------------------------------------

print("\n========== LABEL / GT SOURCE INDICATORS ==========")

indicators = (
    "ground_truth",
    "ground-truth",
    "groundtruth",
    "gt_dir",
    "labels",
    "annotation",
    "evaluate_run",
    "read_csv",
    "open_dataset",
)

for number, line in enumerate(source_lines, start=1):
    if any(term in line.lower() for term in indicators):
        print(f"{number:04d} | {line}")

# This is an indicator search, not proof of absence of GT access.
print("GT_ACCESS_PROVEN_ABSENT=NO")

# ------------------------------------------------------------
# 4. Mounted test/train directory names only
# ------------------------------------------------------------

print("\n========== MOUNTED DATASET NAMES ==========")

for dirname in ("test", "train"):
    parent = DATA_ROOT / dirname

    print("\nDIRECTORY =", parent)
    print("EXISTS =", parent.is_dir())

    if parent.is_dir():
        children = sorted(
            p.name for p in parent.iterdir()
        )

        print("CHILD_COUNT =", len(children))

        for name in children[:50]:
            print("CHILD =", name)

        if len(children) > 50:
            print("ADDITIONAL_CHILDREN =", len(children) - 50)

# ------------------------------------------------------------
# 5. Official GEFF serialization interface
# ------------------------------------------------------------

print("\n========== OFFICIAL SAVE_GRAPH ==========")

print("MODULE =", inspect.getmodule(save_graph).__file__)
print("SIGNATURE =", inspect.signature(save_graph))
print("SOURCE:")
print(inspect.getsource(save_graph))

# ------------------------------------------------------------
# 6. Existing prediction inputs: inventory only
# ------------------------------------------------------------

print("\n========== EXISTING PREDICTION INPUTS ==========")

WORK_ROOT = Path("/kaggle/working/biohub_1415")

for method in (
    "warm2_det0995_v1",
    "warm2_det09975_v1",
):
    directory = (
        WORK_ROOT
        / "predictions"
        / "kaggle_1415"
        / method
        / "split_0"
    )

    assert directory.is_dir(), directory

    names = sorted(
        p.name for p in directory.glob("*.geff")
    )

    print("METHOD =", method)
    print("GEFF_COUNT =", len(names))

    for name in names:
        print("GEFF =", name)

    assert len(names) == 4

print("\n========== 14.15C-1 RESULT ==========")
print("14.15C1_SOURCE_PROVENANCE_AUDIT_COMPLETE")
print("TRACKING_RUN=NO")
print("GT_READ=NO")
print("EVALUATOR_RUN=NO")
print("FILES_WRITTEN=0")

In [ ]:

# ============================================================
# Notebook 14 / 14.15C-2
# Build frozen_v9 policies from the two verified Kaggle
# prediction sets.
#
# Uses test Zarr metadata for voxel scale.
# Does not read train GT or invoke the evaluator.
# Does not modify or save any GEFF.
# ============================================================

import json
import sys
from pathlib import Path

assert sys.platform == "linux"

ROOT = Path(
    "/kaggle/input/datasets/markdd520/"
    "biohub-frozen-v9-warm2-bundle/bundle"
)

DATA_DIR = Path(
    "/kaggle/input/competitions/"
    "biohub-cell-tracking-during-development/test"
)

WORK_ROOT = Path("/kaggle/working/biohub_1415")

SPLITS = WORK_ROOT / "test_splits_1415.json"

PRED_995 = (
    WORK_ROOT / "predictions/kaggle_1415/"
    "warm2_det0995_v1/split_0"
)

PRED_9975 = (
    WORK_ROOT / "predictions/kaggle_1415/"
    "warm2_det09975_v1/split_0"
)

assert ROOT.is_dir()
assert DATA_DIR.is_dir()
assert SPLITS.is_file()
assert PRED_995.is_dir()
assert PRED_9975.is_dir()

sys.dont_write_bytecode = True

paths = [
    ROOT / "external/official/scripts",
    ROOT / "external/official/src",
    ROOT / "src",
    ROOT / "scripts",
]

for path in reversed(paths):
    assert path.is_dir(), path
    value = str(path)
    while value in sys.path:
        sys.path.remove(value)
    sys.path.insert(0, value)

from biohub_cell_tracking import frozen_v9 as frozen

assert Path(frozen.__file__).resolve().is_relative_to(
    ROOT.resolve()
)

dataset_names = json.loads(
    SPLITS.read_text(encoding="utf-8")
)[0]["test"]

actual_names = sorted(
    p.name
    for p in DATA_DIR.iterdir()
    if p.is_dir() and p.name.endswith(".zarr")
)

assert dataset_names == actual_names, (
    "当前测试目录与已冻结的动态测试列表不一致"
)

assert dataset_names
assert len(dataset_names) == len(set(dataset_names))

for name in dataset_names:
    assert (DATA_DIR / name).is_dir(), name
    assert (PRED_995 / f"{name}.geff").is_dir(), name
    assert (PRED_9975 / f"{name}.geff").is_dir(), name

# Explicitly preserve the source-defined prefix behavior.
prefix_map = {
    name: name.split("_")[0]
    for name in dataset_names
}

cfg = frozen.FROZEN_V9_CONFIG

assert cfg.det_threshold == 0.995
assert cfg.high_threshold == 0.9975
assert cfg.confidence_component_limit == 9
assert tuple(cfg.model_downsample_zyx) == (1.0, 4.0, 4.0)

print(
    "========== 14.15C-2 / FROZEN POLICY INPUTS ==========",
    flush=True,
)

print("DATA_DIR =", DATA_DIR, flush=True)
print("PRED_995 =", PRED_995, flush=True)
print("PRED_9975 =", PRED_9975, flush=True)
print("DATASET_COUNT =", len(dataset_names), flush=True)
print("PREFIX_MAP =", prefix_map, flush=True)
print("CONFIG =", cfg, flush=True)
print("GT_DIR_PROVIDED = NO", flush=True)

# Keep the returned object in notebook memory for 14.15C-3.
policies_1415 = frozen.build_frozen_v9_policies(
    dataset_names=dataset_names,
    pred_995_dir=PRED_995,
    pred_9975_dir=PRED_9975,
    train_dir=DATA_DIR,
    prefix_map=prefix_map,
    config=cfg,
    progress=True,
)

print(
    "\n========== 14.15C-2 / POLICY SUMMARY ==========",
    flush=True,
)

tables = {
    "gap_candidates": policies_1415.gap_candidates,
    "single_policy": policies_1415.single_policy,
    "isolated_candidates": policies_1415.isolated_candidates,
    "isolated_scope": policies_1415.isolated_scope,
    "strict_two_policy": policies_1415.strict_two_policy,
}

for name, frame in tables.items():
    print(
        name,
        "ROWS =", len(frame),
        "COLUMNS =", len(frame.columns),
        flush=True,
    )

    if "dataset" in frame.columns:
        print(
            "  PER_DATASET =",
            frame.groupby("dataset").size().to_dict(),
            flush=True,
        )

assert policies_1415.config == cfg

print("\n14.15C2_FROZEN_POLICIES_BUILT")
print("TRACKED_GEFFS_SAVED=0")
print("INPUT_GEFFS_MODIFIED=NO")
print("GT_READ=NO")
print("EVALUATOR_RUN=NO")

In [ ]:

# ============================================================
# Notebook 14 / 14.15C-2
# Build frozen_v9 policies from the two verified Kaggle
# prediction sets.
#
# Uses test Zarr metadata for voxel scale.
# Does not read train GT or invoke the evaluator.
# Does not modify or save any GEFF.
# ============================================================

import json
import sys
from pathlib import Path

assert sys.platform == "linux"

ROOT = Path(
    "/kaggle/input/datasets/markdd520/"
    "biohub-frozen-v9-warm2-bundle/bundle"
)

DATA_DIR = Path(
    "/kaggle/input/competitions/"
    "biohub-cell-tracking-during-development/test"
)

WORK_ROOT = Path("/kaggle/working/biohub_1415")

SPLITS = WORK_ROOT / "test_splits_1415.json"

PRED_995 = (
    WORK_ROOT / "predictions/kaggle_1415/"
    "warm2_det0995_v1/split_0"
)

PRED_9975 = (
    WORK_ROOT / "predictions/kaggle_1415/"
    "warm2_det09975_v1/split_0"
)

assert ROOT.is_dir()
assert DATA_DIR.is_dir()
assert SPLITS.is_file()
assert PRED_995.is_dir()
assert PRED_9975.is_dir()

sys.dont_write_bytecode = True

paths = [
    ROOT / "external/official/scripts",
    ROOT / "external/official/src",
    ROOT / "src",
    ROOT / "scripts",
]

for path in reversed(paths):
    assert path.is_dir(), path
    value = str(path)
    while value in sys.path:
        sys.path.remove(value)
    sys.path.insert(0, value)

from biohub_cell_tracking import frozen_v9 as frozen

assert Path(frozen.__file__).resolve().is_relative_to(
    ROOT.resolve()
)

dataset_names = json.loads(
    SPLITS.read_text(encoding="utf-8")
)[0]["test"]

actual_names = sorted(
    p.name
    for p in DATA_DIR.iterdir()
    if p.is_dir() and p.name.endswith(".zarr")
)

assert dataset_names == actual_names, (
    "当前测试目录与已冻结的动态测试列表不一致"
)

assert dataset_names
assert len(dataset_names) == len(set(dataset_names))

for name in dataset_names:
    assert (DATA_DIR / name).is_dir(), name
    assert (PRED_995 / f"{name}.geff").is_dir(), name
    assert (PRED_9975 / f"{name}.geff").is_dir(), name

# Explicitly preserve the source-defined prefix behavior.
prefix_map = {
    name: name.split("_")[0]
    for name in dataset_names
}

cfg = frozen.FROZEN_V9_CONFIG

assert cfg.det_threshold == 0.995
assert cfg.high_threshold == 0.9975
assert cfg.confidence_component_limit == 9
assert tuple(cfg.model_downsample_zyx) == (1.0, 4.0, 4.0)

print(
    "========== 14.15C-2 / FROZEN POLICY INPUTS ==========",
    flush=True,
)

print("DATA_DIR =", DATA_DIR, flush=True)
print("PRED_995 =", PRED_995, flush=True)
print("PRED_9975 =", PRED_9975, flush=True)
print("DATASET_COUNT =", len(dataset_names), flush=True)
print("PREFIX_MAP =", prefix_map, flush=True)
print("CONFIG =", cfg, flush=True)
print("GT_DIR_PROVIDED = NO", flush=True)

# Keep the returned object in notebook memory for 14.15C-3.
policies_1415 = frozen.build_frozen_v9_policies(
    dataset_names=dataset_names,
    pred_995_dir=PRED_995,
    pred_9975_dir=PRED_9975,
    train_dir=DATA_DIR,
    prefix_map=prefix_map,
    config=cfg,
    progress=True,
)

print(
    "\n========== 14.15C-2 / POLICY SUMMARY ==========",
    flush=True,
)

tables = {
    "gap_candidates": policies_1415.gap_candidates,
    "single_policy": policies_1415.single_policy,
    "isolated_candidates": policies_1415.isolated_candidates,
    "isolated_scope": policies_1415.isolated_scope,
    "strict_two_policy": policies_1415.strict_two_policy,
}

for name, frame in tables.items():
    print(
        name,
        "ROWS =", len(frame),
        "COLUMNS =", len(frame.columns),
        flush=True,
    )

    if "dataset" in frame.columns:
        print(
            "  PER_DATASET =",
            frame.groupby("dataset").size().to_dict(),
            flush=True,
        )

assert policies_1415.config == cfg

print("\n14.15C2_FROZEN_POLICIES_BUILT")
print("TRACKED_GEFFS_SAVED=0")
print("INPUT_GEFFS_MODIFIED=NO")
print("GT_READ=NO")
print("EVALUATOR_RUN=NO")

In [ ]:

# ============================================================
# Notebook 14 / 14.15C-2-R1
# Diagnose Kaggle session state after missing SPLITS.
#
# Read-only. No model inference, tracking, GT, or file writes.
# ============================================================

import sys
import importlib.util
from pathlib import Path
from importlib import metadata

ROOT = Path(
    "/kaggle/input/datasets/markdd520/"
    "biohub-frozen-v9-warm2-bundle/bundle"
)

DATA_DIR = Path(
    "/kaggle/input/competitions/"
    "biohub-cell-tracking-during-development/test"
)

WORK_ROOT = Path("/kaggle/working/biohub_1415")
SPLITS = WORK_ROOT / "test_splits_1415.json"

PRED_995 = (
    WORK_ROOT / "predictions/kaggle_1415/"
    "warm2_det0995_v1/split_0"
)

PRED_9975 = (
    WORK_ROOT / "predictions/kaggle_1415/"
    "warm2_det09975_v1/split_0"
)

print("========== 14.15C-2-R1 / ENVIRONMENT ==========")
print("PYTHON =", sys.executable)
print("PLATFORM =", sys.platform)
print("WORKING_EXISTS =", Path("/kaggle/working").is_dir())
print("SOURCE_EXISTS =", ROOT.is_dir())
print("SOURCE_MANIFEST_EXISTS =", (ROOT / "bundle_manifest.json").is_file())
print("TEST_DATA_EXISTS =", DATA_DIR.is_dir())
print("WORK_ROOT_EXISTS =", WORK_ROOT.is_dir())
print("SPLITS_EXISTS =", SPLITS.is_file())

print("\n========== PACKAGE AVAILABILITY ==========")

for package in ("torch", "zarr", "geff", "tracksdata",
                "polars", "ilpy", "pyscipopt"):
    try:
        version = metadata.version(package)
    except metadata.PackageNotFoundError:
        version = "NOT_INSTALLED"

    print(package, "=", version)

print("\n========== TEST DATA ==========")

test_names = []

if DATA_DIR.is_dir():
    test_names = sorted(
        p.name
        for p in DATA_DIR.iterdir()
        if p.is_dir() and p.name.endswith(".zarr")
    )

print("TEST_COUNT =", len(test_names))
print("TEST_NAMES =", test_names)

print("\n========== PREDICTION ARTIFACTS ==========")

expected = sorted(f"{name}.geff" for name in test_names)

for label, directory in (
    ("DET_0995", PRED_995),
    ("DET_09975", PRED_9975),
):
    exists = directory.is_dir()

    actual = (
        sorted(p.name for p in directory.glob("*.geff"))
        if exists else []
    )

    print(label + "_DIR_EXISTS =", exists)
    print(label + "_GEFF_COUNT =", len(actual))
    print(label + "_GEFF_NAMES =", actual)
    print(label + "_LIST_MATCH =", bool(expected) and actual == expected)

print("\n========== RECOVERY CLASSIFICATION ==========")

predictions_intact = bool(expected) and all(
    directory.is_dir()
    and sorted(p.name for p in directory.glob("*.geff")) == expected
    for directory in (PRED_995, PRED_9975)
)

if not ROOT.is_dir() or not DATA_DIR.is_dir():
    print("RECOVERY_STATUS=INPUT_MOUNT_INCOMPLETE")
elif predictions_intact and not SPLITS.is_file():
    print("RECOVERY_STATUS=ONLY_SPLITS_MISSING")
elif predictions_intact and SPLITS.is_file():
    print("RECOVERY_STATUS=ARTIFACT_PATHS_PRESENT")
else:
    print("RECOVERY_STATUS=PREDICTION_ARTIFACTS_MISSING_OR_INCOMPLETE")

print("\n14.15C2_R1_DIAGNOSTIC_COMPLETE")
print("INFERENCE_RUN=NO")
print("TRACKING_RUN=NO")
print("GT_READ=NO")
print("FILES_WRITTEN=0")

In [ ]:

# ============================================================
# Notebook 14 / 14.15D-1
# Build a Kaggle Linux / Python 3.12 offline dependency overlay.
#
# Downloads pinned wheels and builds tracksdata from the exact
# source commit. Does NOT install, import project code, infer,
# track, evaluate, or read GT.
# ============================================================

import sys
import json
import hashlib
import subprocess
import shutil
import zipfile
from pathlib import Path
from importlib import metadata as importlib_metadata

assert sys.platform == "linux"
assert sys.version_info[:2] == (3, 12), (
    f"Expected Kaggle Python 3.12, got {sys.version}"
)

WORKING = Path("/kaggle/working")
assert WORKING.is_dir()

BUNDLE = Path(
    "/kaggle/input/datasets/markdd520/"
    "biohub-frozen-v9-warm2-bundle/bundle"
)
assert (BUNDLE / "bundle_manifest.json").is_file()

OUT = WORKING / "biohub_wheelhouse_py312_v1"
WHEELS = OUT / "wheels"

ZIP_PATH = WORKING / "biohub_wheelhouse_py312_v1.zip"

# Never overwrite an earlier build.
assert not OUT.exists(), f"Output already exists: {OUT}"
assert not ZIP_PATH.exists(), f"ZIP already exists: {ZIP_PATH}"

TRACKSDATA_COMMIT = (
    "7bfeaf845ceb951226f19b72fe5b80e01601018a"
)

TRACKSDATA_URL = (
    "git+https://github.com/royerlab/tracksdata@"
    + TRACKSDATA_COMMIT
)

# Packages actually added or aligned during the successful
# Kaggle import-repair sequence.
# Base packages such as torch/numpy/pandas are intentionally
# not downloaded here; they will be checked in a clean runtime.

PINNED = [
    "zarr==3.1.6",
    "numcodecs==0.15.1",
    "donfig==0.8.1.post1",
    "geff==1.3.0.1.2",
    "geff-spec==1.2.0",
    "polars==1.43.2",
    "polars-runtime-32==1.43.2",
    "bidict==0.23.1",
    "ilpy==0.6.0",
    "pyscipopt==6.2.1",
    "rustworkx==0.18.1",
    "imagecodecs==2026.3.6",
]

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(
            lambda: stream.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()

def run_checked(command, label):
    print(f"\n========== {label} ==========", flush=True)
    print("COMMAND =", command, flush=True)

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        errors="replace",
    )

    print(result.stdout[-20000:], flush=True)
    print("EXIT_CODE =", result.returncode, flush=True)

    assert result.returncode == 0, (
        f"{label} failed. Stop; preserve the partial build."
    )

OUT.mkdir(parents=False, exist_ok=False)
WHEELS.mkdir(parents=False, exist_ok=False)

# ------------------------------------------------------------
# 1. Download known pinned Linux-compatible binary wheels
# ------------------------------------------------------------

run_checked(
    [
        sys.executable,
        "-m",
        "pip",
        "download",
        "--no-deps",
        "--only-binary=:all:",
        "--dest",
        str(WHEELS),
        *PINNED,
    ],
    "DOWNLOAD PINNED WHEELS",
)

# ------------------------------------------------------------
# 2. Build the exact source-pinned tracksdata wheel
# ------------------------------------------------------------

run_checked(
    [
        sys.executable,
        "-m",
        "pip",
        "wheel",
        "--no-deps",
        "--wheel-dir",
        str(WHEELS),
        TRACKSDATA_URL,
    ],
    "BUILD TRACKSDATA",
)

# ------------------------------------------------------------
# 3. Verify wheel inventory and tracksdata metadata
# ------------------------------------------------------------

wheel_files = sorted(WHEELS.glob("*.whl"))

assert len(wheel_files) == len(PINNED) + 1, (
    f"Expected {len(PINNED) + 1} wheels, "
    f"found {len(wheel_files)}"
)

tracksdata_files = [
    path
    for path in wheel_files
    if path.name.lower().startswith("tracksdata-")
]

assert len(tracksdata_files) == 1, (
    "Expected exactly one tracksdata wheel"
)

tracksdata_wheel = tracksdata_files[0]

with zipfile.ZipFile(tracksdata_wheel) as archive:
    metadata_names = [
        name for name in archive.namelist()
        if name.endswith(".dist-info/METADATA")
    ]

    assert len(metadata_names) == 1

    metadata_text = archive.read(
        metadata_names[0]
    ).decode("utf-8")

expected_version = "0.1.0rc9.dev4+g7bfeaf845"

assert f"Version: {expected_version}" in metadata_text, (
    "tracksdata wheel version does not match the frozen commit"
)

# Record the newly produced wheel identity.
# Do not substitute either historical SHA.
tracksdata_sha = sha256_file(tracksdata_wheel)

print("\n========== TRACKSDATA IDENTITY ==========")
print("GIT_URL =", TRACKSDATA_URL)
print("WHEEL =", tracksdata_wheel.name)
print("VERSION =", expected_version)
print("NEW_WHEEL_SHA256 =", tracksdata_sha)

print("HISTORICAL_KAGGLE_WHEEL_SHA256 =",
      "9e5ecf8cc4e297c55515e836109c34eaee7103a27b73563909db06e0b0bea8c4")

print("LOCAL_CODEX_WHEEL_SHA256 =",
      "a12b65a10ab50f1449c532de72c8bf0c1cf0495088915dc93d4a30db93071df8")

# ------------------------------------------------------------
# 4. Produce an auditable manifest
# ------------------------------------------------------------

wheel_records = []

for path in wheel_files:
    record = {
        "filename": path.name,
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    wheel_records.append(record)

    print(
        "WHEEL",
        record["filename"],
        "BYTES",
        record["size_bytes"],
        "SHA256",
        record["sha256"],
    )

manifest = {
    "stage": "14.15D-1",
    "purpose": "kaggle_py312_offline_dependency_overlay",
    "python": "3.12",
    "platform": "linux_x86_64",
    "source_bundle_manifest_sha256":
        "615c3550cba46a5032344549d2769201af22c0425beb9349726874669ae9c865",
    "tracksdata_git_url": TRACKSDATA_URL,
    "tracksdata_commit": TRACKSDATA_COMMIT,
    "tracksdata_wheel_sha256": tracksdata_sha,
    "pinned_requirements": PINNED,
    "wheels": wheel_records,
    "torch_numpy_pandas_expected_from_base": {
        "torch": "2.10.0+cu128",
        "numpy": "2.0.2",
        "pandas": "2.3.3",
    },
    "status": "BUILT_NOT_OFFLINE_TESTED",
}

manifest_path = OUT / "wheelhouse_manifest.json"

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)

# ------------------------------------------------------------
# 5. Create one portable ZIP for private Kaggle Dataset upload
# ------------------------------------------------------------

created = shutil.make_archive(
    str(ZIP_PATH.with_suffix("")),
    "zip",
    root_dir=OUT,
)

assert Path(created) == ZIP_PATH
assert ZIP_PATH.is_file()

print("\n========== 14.15D-1 RESULT ==========")
print("WHEEL_COUNT =", len(wheel_files))
print("MANIFEST =", manifest_path)
print("MANIFEST_SHA256 =", sha256_file(manifest_path))
print("ZIP =", ZIP_PATH)
print("ZIP_BYTES =", ZIP_PATH.stat().st_size)
print("ZIP_SHA256 =", sha256_file(ZIP_PATH))

print("\n14.15D1_WHEELHOUSE_BUILT")
print("OFFLINE_INSTALL_VERIFIED=NO")
print("GT_READ=NO")
print("INFERENCE_RUN=NO")
print("TRACKING_RUN=NO")

In [ ]:

# Notebook 14 / 14.15D-2
# Independently verify the built wheelhouse ZIP.
# Read-only: no installation, inference, GT, or file writes.

import json
import hashlib
import zipfile
from pathlib import Path

BASE = Path("/kaggle/working")
OUT = BASE / "biohub_wheelhouse_py312_v1"
WHEELS = OUT / "wheels"
MANIFEST = OUT / "wheelhouse_manifest.json"
ZIP = BASE / "biohub_wheelhouse_py312_v1.zip"

EXPECTED_ZIP_SHA = (
    "b25787484b0a6430caaf203aa5557ffefd4"
    "e7713c00b537accaf47c4b0a1f1bc"
)

EXPECTED_MANIFEST_SHA = (
    "9e132365f3c182a34d67a2c6b89d78f0"
    "bce246bf2bec6960d2f4ef824be3aafa"
)

def sha256_stream(stream):
    h = hashlib.sha256()
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        h.update(chunk)
    return h.hexdigest()

def sha256_file(path):
    with path.open("rb") as stream:
        return sha256_stream(stream)

assert ZIP.is_file(), "ZIP 已丢失；停止"
assert MANIFEST.is_file(), "manifest 已丢失；停止"
assert WHEELS.is_dir(), "wheel 目录已丢失；停止"

assert sha256_file(ZIP) == EXPECTED_ZIP_SHA
assert sha256_file(MANIFEST) == EXPECTED_MANIFEST_SHA

manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
records = manifest["wheels"]

assert len(records) == 13
assert manifest["status"] == "BUILT_NOT_OFFLINE_TESTED"

expected_zip_files = {"wheelhouse_manifest.json"} | {
    "wheels/" + record["filename"]
    for record in records
}

with zipfile.ZipFile(ZIP, "r") as archive:
    names = archive.namelist()

    assert len(names) == len(set(names)), "ZIP 存在重复路径"
    assert set(names) == expected_zip_files, "ZIP 文件集合不匹配"

    for record in records:
        filename = record["filename"]
        local = WHEELS / filename
        archived = "wheels/" + filename

        assert local.is_file(), filename
        assert local.stat().st_size == record["size_bytes"]
        assert sha256_file(local) == record["sha256"]

        with archive.open(archived, "r") as stream:
            assert sha256_stream(stream) == record["sha256"], (
                f"ZIP 内 wheel 哈希不匹配：{filename}"
            )

        print("PASS", filename, flush=True)

print("\n========== 14.15D-2 RESULT ==========")
print("WHEEL_COUNT =", len(records))
print("ZIP =", ZIP)
print("ZIP_SHA256 =", sha256_file(ZIP))
print("ZIP_BYTES =", ZIP.stat().st_size)
print("MANIFEST_SHA256 =", sha256_file(MANIFEST))
print("14.15D2_WHEELHOUSE_ZIP_INTEGRITY_PASS")
print("OFFLINE_INSTALL_VERIFIED=NO")
print("INFERENCE_RUN=NO")
print("GT_READ=NO")

In [ ]:

# ============================================================
# Notebook 14 / 14.15D-2-R1
# Corrected read-only wheelhouse ZIP integrity verification.
#
# ZIP directory entries are distinguished from actual files.
# No installation, inference, GT, or file writes.
# ============================================================

import json
import hashlib
import zipfile
from pathlib import Path

BASE = Path("/kaggle/working")
OUT = BASE / "biohub_wheelhouse_py312_v1"
WHEELS = OUT / "wheels"
MANIFEST = OUT / "wheelhouse_manifest.json"
ZIP_PATH = BASE / "biohub_wheelhouse_py312_v1.zip"

EXPECTED_ZIP_SHA = (
    "b25787484b0a6430caaf203aa5557ffefd4"
    "e7713c00b537accaf47c4b0a1f1bc"
)

EXPECTED_MANIFEST_SHA = (
    "9e132365f3c182a34d67a2c6b89d78f0"
    "bce246bf2bec6960d2f4ef824be3aafa"
)

EXPECTED_TRACKSDATA_SHA = (
    "9e5ecf8cc4e297c55515e836109c34e"
    "aee7103a27b73563909db06e0b0bea8c4"
)

def sha256_stream(stream):
    h = hashlib.sha256()
    for chunk in iter(
        lambda: stream.read(1024 * 1024),
        b"",
    ):
        h.update(chunk)
    return h.hexdigest()

def sha256_file(path):
    with path.open("rb") as stream:
        return sha256_stream(stream)

# ------------------------------------------------------------
# 1. Verify outer artifacts
# ------------------------------------------------------------

assert ZIP_PATH.is_file(), "ZIP 不存在，停止"
assert MANIFEST.is_file(), "manifest 不存在，停止"
assert WHEELS.is_dir(), "wheels 目录不存在，停止"

actual_zip_sha = sha256_file(ZIP_PATH)
actual_manifest_sha = sha256_file(MANIFEST)

print("========== OUTER ARTIFACTS ==========")
print("ZIP_SHA256 =", actual_zip_sha)
print("MANIFEST_SHA256 =", actual_manifest_sha)

assert actual_zip_sha == EXPECTED_ZIP_SHA, "ZIP SHA 不匹配"
assert actual_manifest_sha == EXPECTED_MANIFEST_SHA, (
    "manifest SHA 不匹配"
)

manifest = json.loads(
    MANIFEST.read_text(encoding="utf-8")
)

records = manifest["wheels"]

assert len(records) == 13
assert manifest["status"] == "BUILT_NOT_OFFLINE_TESTED"
assert (
    manifest["tracksdata_wheel_sha256"]
    == EXPECTED_TRACKSDATA_SHA
)

expected_files = {
    "wheelhouse_manifest.json"
} | {
    "wheels/" + record["filename"]
    for record in records
}

# ------------------------------------------------------------
# 2. Inspect actual ZIP entries
# ------------------------------------------------------------

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    infos = archive.infolist()
    names = [info.filename for info in infos]

    print("\n========== ZIP INVENTORY ==========")

    assert len(names) == len(set(names)), (
        "ZIP 内存在重复路径"
    )

    directory_entries = {
        info.filename
        for info in infos
        if info.is_dir()
    }

    actual_files = {
        info.filename
        for info in infos
        if not info.is_dir()
    }

    print("DIRECTORY_ENTRIES =", sorted(directory_entries))
    print("FILE_COUNT =", len(actual_files))
    print("EXPECTED_FILE_COUNT =", len(expected_files))

    missing = expected_files - actual_files
    unexpected = actual_files - expected_files

    print("MISSING_FILES =", sorted(missing))
    print("UNEXPECTED_FILES =", sorted(unexpected))

    # The archive builder may include the wheels/ directory.
    # No other directory entries are permitted.
    assert directory_entries <= {"wheels/"}, (
        f"ZIP 包含非预期目录：{directory_entries}"
    )

    assert not missing, "ZIP 缺少必需文件"
    assert not unexpected, "ZIP 包含非预期文件"

    print("\n========== MANIFEST INSIDE ZIP ==========")

    with archive.open(
        "wheelhouse_manifest.json",
        "r",
    ) as stream:
        archived_manifest_sha = sha256_stream(stream)

    print("ARCHIVED_MANIFEST_SHA =", archived_manifest_sha)

    assert archived_manifest_sha == EXPECTED_MANIFEST_SHA, (
        "ZIP 内 manifest 与外部 manifest 不一致"
    )

    # --------------------------------------------------------
    # 3. Verify every wheel: disk vs manifest vs ZIP
    # --------------------------------------------------------

    print("\n========== WHEEL HASH CHECK ==========")

    for record in records:
        filename = record["filename"]
        relative = "wheels/" + filename
        local = WHEELS / filename

        assert local.is_file(), filename

        assert local.stat().st_size == record["size_bytes"], (
            f"本地 wheel 大小不匹配：{filename}"
        )

        local_sha = sha256_file(local)

        assert local_sha == record["sha256"], (
            f"本地 wheel SHA 不匹配：{filename}"
        )

        with archive.open(relative, "r") as stream:
            archived_sha = sha256_stream(stream)

        assert archived_sha == record["sha256"], (
            f"ZIP 内 wheel SHA 不匹配：{filename}"
        )

        print("PASS", filename, flush=True)

# ------------------------------------------------------------
# 4. Final status
# ------------------------------------------------------------

print("\n========== 14.15D-2-R1 RESULT ==========")
print("WHEEL_COUNT =", len(records))
print("ZIP_BYTES =", ZIP_PATH.stat().st_size)
print("ZIP_SHA256 =", actual_zip_sha)
print("MANIFEST_SHA256 =", actual_manifest_sha)

print("14.15D2_R1_WHEELHOUSE_ZIP_INTEGRITY_PASS")
print("OFFLINE_INSTALL_VERIFIED=NO")
print("INFERENCE_RUN=NO")
print("GT_READ=NO")
print("FILES_WRITTEN=0")

In [ ]:

from pathlib import Path

WORKING = Path("/kaggle/working")
INPUT = Path("/kaggle/input")

expected = [
    WORKING / "biohub_wheelhouse_py312_v1.zip",
    WORKING / "biohub_wheelhouse_py312_v1/wheelhouse_manifest.json",
    WORKING / "biohub_wheelhouse_py312_v1/wheels",
]

print("========== WORKING ARTIFACTS ==========")

for path in expected:
    print("EXISTS =", path.exists(), "PATH =", path)

print("\n========== WORKING TOP LEVEL ==========")

if WORKING.is_dir():
    for path in sorted(WORKING.iterdir()):
        print("DIR" if path.is_dir() else "FILE", path.name)

print("\n========== MOUNTED INPUT DATASETS ==========")

dataset_root = INPUT / "datasets" / "markdd520"

if dataset_root.is_dir():
    for dataset in sorted(dataset_root.iterdir()):
        print("DATASET =", dataset.name)

        if dataset.is_dir():
            for item in sorted(dataset.iterdir()):
                print(
                    " ",
                    "DIR" if item.is_dir() else "FILE",
                    item.name,
                )

print("\n14.15D2_R2_ARTIFACT_LOCATION_INSPECTED")
print("FILES_WRITTEN=0")

In [ ]:

from pathlib import Path

WORKING = Path("/kaggle/working")
INPUT = Path("/kaggle/input")

expected = [
    WORKING / "biohub_wheelhouse_py312_v1.zip",
    WORKING / "biohub_wheelhouse_py312_v1/wheelhouse_manifest.json",
    WORKING / "biohub_wheelhouse_py312_v1/wheels",
]

print("========== WORKING ARTIFACTS ==========")

for path in expected:
    print("EXISTS =", path.exists(), "PATH =", path)

print("\n========== WORKING TOP LEVEL ==========")

if WORKING.is_dir():
    for path in sorted(WORKING.iterdir()):
        print("DIR" if path.is_dir() else "FILE", path.name)

print("\n========== MOUNTED INPUT DATASETS ==========")

dataset_root = INPUT / "datasets" / "markdd520"

if dataset_root.is_dir():
    for dataset in sorted(dataset_root.iterdir()):
        print("DATASET =", dataset.name)

        if dataset.is_dir():
            for item in sorted(dataset.iterdir()):
                print(
                    " ",
                    "DIR" if item.is_dir() else "FILE",
                    item.name,
                )

print("\n14.15D2_R2_ARTIFACT_LOCATION_INSPECTED")
print("FILES_WRITTEN=0")